In [ ]:
# Importamos las librerías que vamos a necesitar
# Este bloque de imports va SIEMPRE al principio de cualquier notebook de EDA

import pandas as pd      # Para trabajar con tablas de datos (DataFrames)
import numpy as np       # Para operaciones matemáticas
import matplotlib.pyplot as plt  # Para hacer gráficas estáticas
import seaborn as sns    # Para hacer gráficas estadísticas más profesionales 

# Esta línea hace que las gráficas aparezcan directamente en el notebook
%matplotlib inline

# Configuración visual básica para las gráficas
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Para que no salgan warnings innecesarios que ensucian el output
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [2]:
# Cargamos los 5 datasets de Eurostat
# sep='\t' indica que el separador es una tabulación (formato TSV)
# Eurostat descarga los archivos en este formato por defecto

df_salario    = pd.read_csv('./data/salario_minimo.tsv',   sep='\t')
df_pib        = pd.read_csv('./data/pib_percapita.tsv',    sep='\t')
df_vivienda   = pd.read_csv('./data/precios_vivienda.tsv', sep='\t')
df_gini       = pd.read_csv('./data/gini.tsv',             sep='\t')
df_genero     = pd.read_csv('./data/brecha_genero.tsv',    sep='\t')

print("Los 5 datasets han sido cargados correctamente")

Los 5 datasets han sido cargados correctamente


In [3]:
# DATASET 1: Salario mínimo
# Siempre empezamos con las mismas 4 preguntas para cualquier dataset nuevo:
# 1. ¿Cuántas filas y columnas tiene?
# 2. ¿Qué tipos de datos tiene cada columna?
# 3. ¿Cómo son las primeras filas?
# 4. ¿Cuántos valores nulos hay?

print("=" * 60)
print("DATASET 1: Salario mínimo (earn_mw_cur)")
print("=" * 60)

print("\n--- Dimensiones ---")
print(f"Filas: {df_salario.shape[0]} | Columnas: {df_salario.shape[1]}")

print("\n--- Tipos de datos e información general ---")
print(df_salario.info())

print("\n--- Primeras 5 filas ---")
df_salario.head()

DATASET 1: Salario mínimo (earn_mw_cur)

--- Dimensiones ---
Filas: 38 | Columnas: 11

--- Tipos de datos e información general ---
<class 'pandas.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   freq,currency,geo\TIME_PERIOD  38 non-null     str  
 1   2021-S2                        38 non-null     str  
 2   2022-S1                        38 non-null     str  
 3   2022-S2                        38 non-null     str  
 4   2023-S1                        38 non-null     str  
 5   2023-S2                        38 non-null     str  
 6   2024-S1                        38 non-null     str  
 7   2024-S2                        38 non-null     str  
 8   2025-S1                        38 non-null     str  
 9   2025-S2                        38 non-null     str  
 10  2026-S1                        38 non-null     str  
dtypes: str(11)
memory

,"freq,currency,geo\TIME_PERIOD",2021-S2,2022-S1,2022-S2,2023-S1,2023-S2,2024-S1,2024-S2,2025-S1,2025-S2,2026-S1
0,"S,EUR,AL",245,248,269,298,376,385,399,408,408,517
1,"S,EUR,AT",: m,: m,: m,: m,: m,: m,: m,: m,: m,: m
2,"S,EUR,BE",1626,1658,1842,1955,1955,1994,2070,2070,2112,2112
3,"S,EUR,BG",332,332,363,399,399,477,477,551,551,620
4,"S,EUR,CH",: m,: m,: m,: m,: m,: m,: m,: m,: m,: m


In [5]:
# DATASETS 2 al 5: vistazo rápido al resto
for nombre, df in [
    ("PIB per cápita (tec00114)",        df_pib),
    ("Precios vivienda (prc_hpi_a)",      df_vivienda),
    ("Gini (ilc_di12)",                  df_gini),
    ("Brecha de género (sdg_05_20)",     df_genero)
]:
    print("=" * 60)
    print(f"DATASET: {nombre}")
    print("=" * 60)
    print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
    print(f"Tipos: {df.dtypes.value_counts().to_dict()}")
    print(f"Nulos: {df.isnull().sum().sum()} valores nulos totales")
    print(f"Primera columna: {df.columns[0]}")
    print()

DATASET: PIB per cápita (tec00114)
Filas: 42 | Columnas: 13
Tipos: {dtype('int64'): 9, <StringDtype(storage='python', na_value=nan)>: 4}
Nulos: 0 valores nulos totales
Primera columna: freq,indic_ppp,ppp_cat18,geo\TIME_PERIOD

DATASET: Precios vivienda (prc_hpi_a)
Filas: 330 | Columnas: 11
Tipos: {<StringDtype(storage='python', na_value=nan)>: 11}
Nulos: 0 valores nulos totales
Primera columna: freq,purchase,unit,geo\TIME_PERIOD

DATASET: Gini (ilc_di12)
Filas: 78 | Columnas: 11
Tipos: {<StringDtype(storage='python', na_value=nan)>: 10, dtype('float64'): 1}
Nulos: 0 valores nulos totales
Primera columna: freq,age,statinfo,geo\TIME_PERIOD

DATASET: Brecha de género (sdg_05_20)
Filas: 40 | Columnas: 21
Tipos: {<StringDtype(storage='python', na_value=nan)>: 21}
Nulos: 0 valores nulos totales
Primera columna: freq,unit,nace_r2,geo\TIME_PERIOD



In [8]:
# LIMPIEZA - DATASET 1: Salario mínimo
# Trabajamos siempre sobre una copia para no perder el original
# Si algo sale mal, podemos volver a df_salario sin recargar el archivo

df_sal = df_salario.copy()

# PASO 1: Separar la primera columna
# La columna se llama "freq,currency,geo\TIME_PERIOD"
# Contiene valores como "S,EUR,ES" - separados por comas
# .str.split(',') divide el texto por las comas -> ['S', 'EUR', 'ES']
# .str[-1] coge el último elemento de cada lista -> 'ES'
# .str.strip() elimina espacios en blanco que pueda haber

df_sal['geo'] = df_sal.iloc[:, 0].str.split(',').str[-1].str.strip()

# Verificamos que funciona bien
print("Primeros valores de 'geo':")
print(df_sal['geo'].values[:30])

Primeros valores de 'geo':
['AL' 'AT' 'BE' 'BG' 'CH' 'CY' 'CZ' 'DE' 'DK' 'EE' 'EL' 'ES' 'FI' 'FR'
 'HR' 'HU' 'IE' 'IS' 'IT' 'LT' 'LU' 'LV' 'MD' 'ME' 'MK' 'MT' 'NL' 'NO'
 'PL' 'PT']


In [9]:
# PASO 2: Convertir de formato wide a long con pd.melt()
# Antes:  cada periodo de tiempo es una columna (formato wide)
# Después: cada fila es un pais en un periodo concreto (formato long)

# Primero identificamos las columnas de tiempo
# Son todas las columnas EXCEPTO la primera (que tiene los codigos comprimidos)
# y la nueva columna 'geo' que acabamos de crear
columnas_tiempo = [col for col in df_sal.columns
                   if col not in ['geo', df_sal.columns[0]]]

print("Columnas de tiempo detectadas:")
print(columnas_tiempo)
print(f"Total: {len(columnas_tiempo)} periodos")

Columnas de tiempo detectadas:
['2021-S2 ', '2022-S1 ', '2022-S2 ', '2023-S1 ', '2023-S2 ', '2024-S1 ', '2024-S2 ', '2025-S1 ', '2025-S2 ', '2026-S1 ']
Total: 10 periodos


In [ ]:
''' Aquí arriba vemos que hay columnas que tienen espacios en blanco, 
lo que puede causar problemas al convertir a formato long.
Para evitar problemas, vamos a renombrar las columnas de tiempo con 
.str.strip() le aplicamos la función strip() (que elimina espacios) '''

In [11]:
# Limpiamos los espacios en los nombres de las columnas de tiempo
# .str.strip() elimina espacios al principio y al final de cada nombre
df_sal.columns = df_sal.columns.str.strip()

# Recalculamos las columnas de tiempo ya limpias
columnas_tiempo = [col for col in df_sal.columns
                   if col not in ['geo', df_sal.columns[0]]]

# PASO 2: pd.melt() - de wide a long
# id_vars: columna que se queda fija (el pais)
# value_vars: columnas que se convierten en filas (los periodos)
# var_name: nombre que le damos a la nueva columna de periodos
# value_name: nombre que le damos a la nueva columna de valores

df_sal_long = pd.melt(
    df_sal,
    id_vars='geo',
    value_vars=columnas_tiempo,
    var_name='periodo',
    value_name='salario_minimo_eur'
)

print(f"Filas antes del melt: {df_sal.shape[0]}")
print(f"Filas después del melt: {df_sal_long.shape[0]}")
print()
print("Primeras 10 filas:")
print(df_sal_long.head(10))

Filas antes del melt: 38
Filas después del melt: 380

Primeras 10 filas:
  geo  periodo salario_minimo_eur
0  AL  2021-S2               245 
1  AT  2021-S2                : m
2  BE  2021-S2              1626 
3  BG  2021-S2               332 
4  CH  2021-S2                : m
5  CY  2021-S2                : m
6  CZ  2021-S2               596 
7  DE  2021-S2              1602 
8  DK  2021-S2                : m
9  EE  2021-S2               584 


In [13]:
# PASO 3: Convertir salario_minimo_eur a número
# pd.to_numeric convierte texto a número
# errors='coerce' convierte lo que no pueda convertir en NaN
# (": m", "245 ", etc.)

df_sal_long['salario_minimo_eur'] = pd.to_numeric(
    df_sal_long['salario_minimo_eur'],
    errors='coerce'
)

# Comprobamos cuántos NaN hay ahora
nulos = df_sal_long['salario_minimo_eur'].isnull().sum()
total = len(df_sal_long)

print(f"Valores nulos: {nulos} de {total} ({round(nulos/total*100, 2)}%)")
print()
print(df_sal_long.dtypes)
print()
print(df_sal_long.head(10))

Valores nulos: 83 de 380 (21.84%)

geo                       str
periodo                   str
salario_minimo_eur    float64
dtype: object

  geo  periodo  salario_minimo_eur
0  AL  2021-S2               245.0
1  AT  2021-S2                 NaN
2  BE  2021-S2              1626.0
3  BG  2021-S2               332.0
4  CH  2021-S2                 NaN
5  CY  2021-S2                 NaN
6  CZ  2021-S2               596.0
7  DE  2021-S2              1602.0
8  DK  2021-S2                 NaN
9  EE  2021-S2               584.0


In [ ]:
'''
Los ": m" son países sin salario mínimo (Dinamarca, Austria, Suecia, Suiza...)
y que tenemos 10 períodos para cada uno. Si son unos 8 países sin salario mínimo, 
serían aproximadamente 80 nulos de 380 filas, algo así como un 20%.
Eso no es un problema, es información. Esos nulos nos dicen qué países no tienen 
salario mínimo legal, que es relevante para el análisis.
'''

In [14]:
# PASO 4: Extraer el año del periodo y limpiar
# 'periodo' tiene valores como '2021-S2', '2022-S1'
# Necesitamos el año como número para poder hacer el merge con otros datasets
# .str[:4] coge los primeros 4 caracteres -> '2021'
# astype(int) lo convierte de texto a entero

df_sal_long['anio'] = df_sal_long['periodo'].str[:4].astype(int)

# PASO 5: Eliminar duplicados
df_sal_long = df_sal_long.drop_duplicates()

# PASO 6: Eliminar nulos de la columna principal
# Los paises sin salario minimo no nos sirven para el análisis
df_sal_long = df_sal_long.dropna(subset=['salario_minimo_eur'])

# Resultado final
print("Dataset de salario minimo limpio:")
print(f"Filas: {df_sal_long.shape[0]}")
print(f"Columnas: {df_sal_long.columns.tolist()}")
print()
print(df_sal_long.head(10))

Dataset de salario minimo limpio:
Filas: 297
Columnas: ['geo', 'periodo', 'salario_minimo_eur', 'anio']

   geo  periodo  salario_minimo_eur  anio
0   AL  2021-S2               245.0  2021
2   BE  2021-S2              1626.0  2021
3   BG  2021-S2               332.0  2021
6   CZ  2021-S2               596.0  2021
7   DE  2021-S2              1602.0  2021
9   EE  2021-S2               584.0  2021
10  EL  2021-S2               758.0  2021
11  ES  2021-S2              1108.0  2021
13  FR  2021-S2              1555.0  2021
14  HR  2021-S2               567.0  2021


In [15]:
# LIMPIEZA - DATASET 2: PIB per cápita
df_pib2 = df_pib.copy()

# Limpiamos espacios en nombres de columnas
df_pib2.columns = df_pib2.columns.str.strip()

# PASO 1: Extraer código de país
df_pib2['geo'] = df_pib2.iloc[:, 0].str.split(',').str[-1].str.strip()

# PASO 2: Columnas de tiempo
columnas_tiempo_pib = [col for col in df_pib2.columns
                       if col not in ['geo', df_pib2.columns[0]]]

# PASO 3: Melt
df_pib_long = pd.melt(
    df_pib2,
    id_vars='geo',
    value_vars=columnas_tiempo_pib,
    var_name='anio',
    value_name='pib_pps'
)

# PASO 4: Convertir año a entero y valor a número
df_pib_long['anio'] = pd.to_numeric(df_pib_long['anio'], errors='coerce')
df_pib_long['pib_pps'] = pd.to_numeric(df_pib_long['pib_pps'], errors='coerce')

# PASO 5: Eliminar duplicados y nulos
df_pib_long = df_pib_long.drop_duplicates()
df_pib_long = df_pib_long.dropna(subset=['pib_pps'])

print("Dataset PIB per cápita limpio:")
print(f"Filas: {df_pib_long.shape[0]}")
print(f"Columnas: {df_pib_long.columns.tolist()}")
print()
print(df_pib_long.head(8))

Dataset PIB per cápita limpio:
Filas: 482
Columnas: ['geo', 'anio', 'pib_pps']

  geo  anio  pib_pps
0  AL  2013     29.0
1  AT  2013    131.0
2  BA  2013     30.0
3  BE  2013    121.0
4  BG  2013     46.0
5  CH  2013    173.0
6  CY  2013     84.0
7  CZ  2013     85.0


In [16]:
# LIMPIEZA - DATASET 3: Precios de vivienda
df_viv2 = df_vivienda.copy()
df_viv2.columns = df_viv2.columns.str.strip()
df_viv2['geo'] = df_viv2.iloc[:, 0].str.split(',').str[-1].str.strip()

columnas_tiempo_viv = [col for col in df_viv2.columns
                       if col not in ['geo', df_viv2.columns[0]]]

df_viv_long = pd.melt(
    df_viv2,
    id_vars='geo',
    value_vars=columnas_tiempo_viv,
    var_name='anio',
    value_name='indice_vivienda'
)

df_viv_long['anio'] = pd.to_numeric(df_viv_long['anio'], errors='coerce')
df_viv_long['indice_vivienda'] = pd.to_numeric(df_viv_long['indice_vivienda'], errors='coerce')
df_viv_long = df_viv_long.drop_duplicates()
df_viv_long = df_viv_long.dropna(subset=['indice_vivienda'])

print(f"Vivienda limpio: {df_viv_long.shape[0]} filas | {df_viv_long.columns.tolist()}")
print(df_viv_long.head(5))

Vivienda limpio: 2980 filas | ['geo', 'anio', 'indice_vivienda']
  geo  anio  indice_vivienda
0  AT  2015           128.63
1  BE  2015           108.42
4  CZ  2015           105.40
5  DE  2015           119.20
7  EA  2015            99.20


In [17]:
# LIMPIEZA - DATASET 4: Gini
df_gin2 = df_gini.copy()
df_gin2.columns = df_gin2.columns.str.strip()
df_gin2['geo'] = df_gin2.iloc[:, 0].str.split(',').str[-1].str.strip()

columnas_tiempo_gin = [col for col in df_gin2.columns
                       if col not in ['geo', df_gin2.columns[0]]]

df_gin_long = pd.melt(
    df_gin2,
    id_vars='geo',
    value_vars=columnas_tiempo_gin,
    var_name='anio',
    value_name='gini'
)

df_gin_long['anio'] = pd.to_numeric(df_gin_long['anio'], errors='coerce')
df_gin_long['gini'] = pd.to_numeric(df_gin_long['gini'], errors='coerce')
df_gin_long = df_gin_long.drop_duplicates()
df_gin_long = df_gin_long.dropna(subset=['gini'])

print(f"Gini limpio: {df_gin_long.shape[0]} filas | {df_gin_long.columns.tolist()}")
print(df_gin_long.head(5))

Gini limpio: 639 filas | ['geo', 'anio', 'gini']
  geo  anio  gini
1  AT  2016  27.2
2  BE  2016  26.3
4  CH  2016  29.4
5  CY  2016  32.1
6  CZ  2016  25.1


In [18]:
# LIMPIEZA - DATASET 5: Brecha de género
df_gen2 = df_genero.copy()
df_gen2.columns = df_gen2.columns.str.strip()
df_gen2['geo'] = df_gen2.iloc[:, 0].str.split(',').str[-1].str.strip()

columnas_tiempo_gen = [col for col in df_gen2.columns
                       if col not in ['geo', df_gen2.columns[0]]]

df_gen_long = pd.melt(
    df_gen2,
    id_vars='geo',
    value_vars=columnas_tiempo_gen,
    var_name='anio',
    value_name='brecha_genero'
)

df_gen_long['anio'] = pd.to_numeric(df_gen_long['anio'], errors='coerce')
df_gen_long['brecha_genero'] = pd.to_numeric(df_gen_long['brecha_genero'], errors='coerce')
df_gen_long = df_gen_long.drop_duplicates()
df_gen_long = df_gen_long.dropna(subset=['brecha_genero'])

print(f"Genero limpio: {df_gen_long.shape[0]} filas | {df_gen_long.columns.tolist()}")
print(df_gen_long.head(5))

Genero limpio: 531 filas | ['geo', 'anio', 'brecha_genero']
   geo  anio  brecha_genero
3   BG  2002           18.9
5   CY  2002           22.5
6   CZ  2002           22.1
12  EL  2002           25.5
13  ES  2002           20.2


In [19]:
# Lista oficial de los 27 países de la Unión Europea
# Usamos los códigos ISO de 2 letras que usa Eurostat
# Es importante tenerla definida aquí para reutilizarla en todo el proyecto

EU27 = [
    'AT',  # Austria
    'BE',  # Bélgica
    'BG',  # Bulgaria
    'CY',  # Chipre
    'CZ',  # Chequia
    'DE',  # Alemania
    'DK',  # Dinamarca
    'EE',  # Estonia
    'EL',  # Grecia (Eurostat usa EL, no GR)
    'ES',  # España
    'FI',  # Finlandia
    'FR',  # Francia
    'HR',  # Croacia
    'HU',  # Hungría
    'IE',  # Irlanda
    'IT',  # Italia
    'LT',  # Lituania
    'LU',  # Luxemburgo
    'LV',  # Letonia
    'MT',  # Malta
    'NL',  # Países Bajos
    'PL',  # Polonia
    'PT',  # Portugal
    'RO',  # Rumanía
    'SE',  # Suecia
    'SI',  # Eslovenia
    'SK',  # Eslovaquia
]

print(f"Países UE27 definidos: {len(EU27)}")
print(EU27)

Países UE27 definidos: 27
['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']


In [ ]:
# Preparamos el salario mínimo para el merge
# Necesitamos un solo valor por país y año (no dos semestres)
# Calculamos la media de S1 y S2 para cada año

df_sal_anual = df_sal_long.groupby(['geo', 'anio'])['salario_minimo_eur'].mean().reset_index()

print("Salario mínimo anualizado:")
print(f"Filas: {df_sal_anual.shape[0]}")
print(df_sal_anual[df_sal_anual['geo'] == 'ES']) # Miramos España

Salario mínimo anualizado:
Filas: 178
   geo  anio  salario_minimo_eur
46  ES  2021              1108.0
47  ES  2022              1146.5
48  ES  2023              1213.5
49  ES  2024              1323.0
50  ES  2025              1381.0
51  ES  2026              1381.0


In [23]:
# MERGE de los 5 datasets
# Unimos uno a uno usando geo y anio como clave
# how='outer' conserva todas las filas aunque falte algún dato
# Después filtramos a EU27 y años 2016-2024

# Merge 1: salario + PIB
df_merge = pd.merge(df_sal_anual, df_pib_long, on=['geo', 'anio'], how='outer')

# Merge 2: + vivienda
df_merge = pd.merge(df_merge, df_viv_long, on=['geo', 'anio'], how='outer')

# Merge 3: + gini
df_merge = pd.merge(df_merge, df_gin_long, on=['geo', 'anio'], how='outer')

# Merge 4: + brecha de género
df_merge = pd.merge(df_merge, df_gen_long, on=['geo', 'anio'], how='outer')

print("DataFrame antes de filtrar:")
print(f"Filas: {df_merge.shape[0]} | Columnas: {df_merge.shape[1]}")
print(f"Columnas: {df_merge.columns.tolist()}")
print()

# Filtramos a EU27 y años 2016-2024
df_final = df_merge[
    (df_merge['geo'].isin(EU27)) &
    (df_merge['anio'] >= 2016) &
    (df_merge['anio'] <= 2024)
].copy()

df_final = df_final.reset_index(drop=True)

print("DataFrame final (EU27, 2016-2024):")
print(f"Filas: {df_final.shape[0]} | Columnas: {df_final.shape[1]}")
print()
print(df_final.head(40))

DataFrame antes de filtrar:
Filas: 5695 | Columnas: 7
Columnas: ['geo', 'anio', 'salario_minimo_eur', 'pib_pps', 'indice_vivienda', 'gini', 'brecha_genero']

DataFrame final (EU27, 2016-2024):
Filas: 3972 | Columnas: 7

   geo  anio  salario_minimo_eur  pib_pps  indice_vivienda  gini  \
0   AT  2016                 NaN    128.0           136.27  27.2   
1   AT  2016                 NaN    128.0           136.27  25.9   
2   AT  2016                 NaN    128.0           105.95  27.2   
3   AT  2016                 NaN    128.0           105.95  25.9   
4   AT  2016                 NaN    128.0             5.90  27.2   
5   AT  2016                 NaN    128.0             5.90  25.9   
6   AT  2016                 NaN    128.0           141.62  27.2   
7   AT  2016                 NaN    128.0           141.62  25.9   
8   AT  2016                 NaN    128.0           107.85  27.2   
9   AT  2016                 NaN    128.0           107.85  25.9   
10  AT  2016                 NaN

## Diagnóstico del merge (por qué me ha salido mal): producto cartesiano

Al hacer el merge, el DataFrame resultó con 3972 filas. Eso es demasiado.
El problema es que algunos datasets tienen **varias filas por país y año**
(distintas categorías de vivienda, distintos grupos en Gini...).

Cuando Pandas encuentra, por ejemplo, 9 filas de vivienda y 2 filas de Gini
para Austria-2016, las combina todas con todas: 9 × 2 = 18 filas.
Esto se llama **producto cartesiano** (gracias IA/Claude) y es el error más común en merges con datos de Eurostat por lo que he podido leer. 

Antes de repetir el merge, necesitamos que cada dataset tenga **exactamente una fila por país y año**.

In [22]:
# Ver cuántas filas tiene cada dataset limpio antes del merge
print("Filas por dataset:")
print(f"  Salarios: {len(df_sal_anual)}")
print(f"  PIB:      {len(df_pib_long)}")
print(f"  Vivienda: {len(df_viv_long)}")
print(f"  Gini:     {len(df_gin_long)}")
print(f"  Brecha:   {len(df_gen_long)}")

print("\n--- Vivienda: categorías únicas ---")
print(df_viv_long.columns.tolist())
print(df_viv_long.head(15))

print("\n--- Gini: valores únicos por país/año ---")
print(df_gin_long.groupby(['geo', 'anio']).size().value_counts())

Filas por dataset:
  Salarios: 178
  PIB:      482
  Vivienda: 2980
  Gini:     639
  Brecha:   531

--- Vivienda: categorías únicas ---
['geo', 'anio', 'indice_vivienda']
          geo  anio  indice_vivienda
0          AT  2015           128.63
1          BE  2015           108.42
4          CZ  2015           105.40
5          DE  2015           119.20
7          EA  2015            99.20
8        EA19  2015            99.35
9        EA20  2015            99.29
10       EA21  2015            99.21
11         EE  2015           148.70
12         ES  2015            71.09
13         EU  2015           102.38
14  EU27_2020  2015           100.00
15       EU28  2015           102.37
17         FR  2015            99.42
18         HR  2015            89.95

--- Gini: valores únicos por país/año ---
2    319
1      1
Name: count, dtype: int64


In [24]:
# ============================================================
# PASO 1: Investigar qué categorías tiene el dataset de vivienda
# ============================================================
# El dataset original de vivienda tiene una primera columna compuesta
# tipo "freq,purchase,unit,geo\time" que contiene categorías extra.
# Esas categorías (tipo de vivienda, tipo de índice) son las que
# generan varias filas por país/año.
#
# Necesitamos ver qué hay ahí para decidir con qué nos quedamos.
# ============================================================

print("=== Dataset de vivienda: primeras filas del original ===")
print(df_vivienda.iloc[:5, :5])

print("\n=== Valores únicos en la primera columna ===")
print(df_vivienda.iloc[:, 0].unique())

=== Dataset de vivienda: primeras filas del original ===
  freq,purchase,unit,geo\TIME_PERIOD    2015     2016     2017     2018 
0             A,DW_EXST,I10_A_AVG,AT  128.63   136.27   144.48   153.57 
1             A,DW_EXST,I10_A_AVG,BE  108.42   110.71   114.46   117.96 
2             A,DW_EXST,I10_A_AVG,BG  92.29 b   99.07   108.70   116.35 
3             A,DW_EXST,I10_A_AVG,CY  91.93 b   91.65    91.62    90.26 
4             A,DW_EXST,I10_A_AVG,CZ  105.40   113.10   126.00   136.30 

=== Valores únicos en la primera columna ===
<StringArray>
[  'A,DW_EXST,I10_A_AVG,AT',   'A,DW_EXST,I10_A_AVG,BE',
   'A,DW_EXST,I10_A_AVG,BG',   'A,DW_EXST,I10_A_AVG,CY',
   'A,DW_EXST,I10_A_AVG,CZ',   'A,DW_EXST,I10_A_AVG,DE',
   'A,DW_EXST,I10_A_AVG,DK',   'A,DW_EXST,I10_A_AVG,EA',
 'A,DW_EXST,I10_A_AVG,EA19', 'A,DW_EXST,I10_A_AVG,EA20',
 ...
     'A,TOTAL,RCH_A_AVG,NL',     'A,TOTAL,RCH_A_AVG,NO',
     'A,TOTAL,RCH_A_AVG,PL',     'A,TOTAL,RCH_A_AVG,PT',
     'A,TOTAL,RCH_A_AVG,RO',     'A,TOTAL

In [25]:
# Solo el diagnóstico de Gini
print("--- Gini: filas por país/año ---")
print(df_gin_long.groupby(['geo', 'anio']).size().value_counts())

print("\n--- Gini: primeras filas del original ---")
print(df_gini.iloc[:5, :5])
print("\n--- Valores únicos en primera columna de Gini ---")
print(df_gini.iloc[:, 0].unique())

--- Gini: filas por país/año ---
2    319
1      1
Name: count, dtype: int64

--- Gini: primeras filas del original ---
  freq,age,statinfo,geo\TIME_PERIOD   2016   2017   2018    2019 
0               A,TOTAL,GINI_HND,AL      :   36.8    35.4   34.3 
1               A,TOTAL,GINI_HND,AT   27.2   27.9    26.8   27.5 
2               A,TOTAL,GINI_HND,BE   26.3   26.1    25.7  25.1 b
3               A,TOTAL,GINI_HND,BG  37.7 b  40.2    39.6   40.8 
4               A,TOTAL,GINI_HND,CH   29.4   30.1    29.7   30.6 

--- Valores únicos en primera columna de Gini ---
<StringArray>
[        'A,TOTAL,GINI_HND,AL',         'A,TOTAL,GINI_HND,AT',
         'A,TOTAL,GINI_HND,BE',         'A,TOTAL,GINI_HND,BG',
         'A,TOTAL,GINI_HND,CH',         'A,TOTAL,GINI_HND,CY',
         'A,TOTAL,GINI_HND,CZ',         'A,TOTAL,GINI_HND,DE',
         'A,TOTAL,GINI_HND,DK',       'A,TOTAL,GINI_HND,EA20',
         'A,TOTAL,GINI_HND,EE',         'A,TOTAL,GINI_HND,EL',
         'A,TOTAL,GINI_HND,ES',  'A,TOTAL

In [ ]:
# Ver cuántas filas tiene cada dataset limpio antes del merge
print("Filas por dataset:")
print(f"  Salarios: {len(df_sal_anual)}")
print(f"  PIB:      {len(df_pib_long)}")
print(f"  Vivienda: {len(df_viv_long)}")
print(f"  Gini:     {len(df_gin_long)}")
print(f"  Brecha:   {len(df_gen_long)}")

print("\n--- Vivienda: categorías únicas ---")
print(df_viv_long.columns.tolist())
print(df_viv_long.head(15))

print("\n--- Gini: valores únicos por país/año ---")
print(df_gin_long.groupby(['geo', 'anio']).size().value_counts())

Filas por dataset:
  Salarios: 178
  PIB:      482
  Vivienda: 2980
  Gini:     639
  Brecha:   531

--- Vivienda: categorías únicas ---
['geo', 'anio', 'indice_vivienda']
          geo  anio  indice_vivienda
0          AT  2015           128.63
1          BE  2015           108.42
4          CZ  2015           105.40
5          DE  2015           119.20
7          EA  2015            99.20
8        EA19  2015            99.35
9        EA20  2015            99.29
10       EA21  2015            99.21
11         EE  2015           148.70
12         ES  2015            71.09
13         EU  2015           102.38
14  EU27_2020  2015           100.00
15       EU28  2015           102.37
17         FR  2015            99.42
18         HR  2015            89.95

--- Gini: valores únicos por país/año ---
2    319
1      1
Name: count, dtype: int64


## Diagnóstico: ¿por qué el merge genera filas duplicadas?

Al investigar los datasets originales, descubrimos que **dos de los cinco** tienen
varias filas por país y año:

### Dataset de vivienda (`precios_vivienda.tsv`)
La primera columna contiene valores como `A,DW_EXST,I10_A_AVG,AT`, donde:
- `DW_EXST` = vivienda existente, `DW_NEW` = vivienda nueva, `TOTAL` = ambas
- `I10_A_AVG` = índice con base 2010 (cuánto han subido los precios), `RCH_A_AVG` = tasa de cambio anual (% de subida respecto al año anterior)

Cada combinación (tipo de vivienda × tipo de índice) genera una fila distinta.
Por ejemplo, Austria 2016 tiene **9 filas** en vivienda.

### Dataset de Gini (`gini.tsv`)
La primera columna contiene valores como `A,TOTAL,GINI_HND,AT` y `A,Y_LT18,GINI_HND,AT`, donde:
- `TOTAL` = Gini calculado sobre toda la población
- `Y_LT18` = Gini calculado solo sobre menores de 18 años

Esto genera **2 filas** por país y año.

### El producto cartesiano
Cuando Pandas hace el merge por `geo + anio`, encuentra 9 filas de vivienda
y 2 filas de Gini para Austria-2016. Como no sabe cuál emparejar con cuál,
las combina **todas con todas**: 9 × 2 = 18 filas para un solo país y año.

### La solución
Antes del merge, necesitamos **filtrar** cada dataset para quedarnos con una sola
categoría por país y año:
- **Vivienda**: nos quedamos con `TOTAL` (todo tipo de vivienda) + `I10_A_AVG` (índice base 2010), que es lo más útil para la H3 (¿el salario alcanza para pagar vivienda?)
- **Gini**: nos quedamos con `TOTAL` (toda la población), que es el indicador estándar de desigualdad

### ¿Por qué elegimos estas categorías?

**Vivienda: `TOTAL` + `I10_A_AVG`**
- `TOTAL` porque no queremos diferenciar entre vivienda nueva y existente; nos interesa el mercado completo
- `I10_A_AVG` (índice base 2010 = 100) porque nos permite comparar cuánto han subido los precios en cada país. Si España tiene 150 y Alemania tiene 120, sabemos que los precios españoles han subido un 50% desde 2010 frente al 20% de Alemania. La tasa de cambio anual (`RCH_A_AVG`) solo nos dice el % de un año a otro, que es menos útil para nuestro análisis

**Gini: `TOTAL`**
- El coeficiente de Gini estándar se calcula sobre toda la población. El de menores de 18 (`Y_LT18`) es un indicador específico de pobreza infantil que no necesitamos para nuestras hipótesis

### Relimpieza de vivienda y Gini

Volvemos al dataset **original** de cada uno y repetimos la limpieza,
pero esta vez extrayendo la categoría de la primera columna compuesta
para poder filtrar antes del `melt()`.

La estrategia es:
1. Separar la primera columna compuesta en sus partes (freq, categoría, unidad, geo)
2. Filtrar solo las filas de la categoría que nos interesa
3. Hacer el `melt()` ya con una sola fila por país/año

In [ ]:
# ============================================================
# RELIMPIEZA DE VIVIENDA: filtrar ANTES del melt
# ============================================================
# La primera columna tiene formato: "A,DW_EXST,I10_A_AVG,AT"
#                                    freq,purchase,unit,geo
#
# Paso 1: Extraer las partes de la columna compuesta
# .str.split(',') parte el texto por las comas
# .expand=True crea una columna separada para cada parte
# ============================================================

col_compuesta = df_vivienda.columns[0]  # nombre de la primera columna

partes = df_vivienda[col_compuesta].str.split(',', expand=True)
# partes tiene ahora 4 columnas: [0]=freq, [1]=purchase, [2]=unit, [3]=geo (que es AT ej)

# Asignamos nombres claros
df_vivienda['freq'] = partes[0]
df_vivienda['purchase'] = partes[1]    # tipo de vivienda
df_vivienda['unit'] = partes[2]        # tipo de índice
df_vivienda['geo'] = partes[3]         # código de país

# Veamos qué combinaciones existen
print("Tipos de vivienda (purchase):")
print(df_vivienda['purchase'].unique())
print("\nTipos de índice (unit):")
print(df_vivienda['unit'].unique())
print("\nCombinaciones y cuántas filas tiene cada una:")
print(df_vivienda.groupby(['purchase', 'unit']).size())

Tipos de vivienda (purchase):
<StringArray>
['DW_EXST', 'DW_NEW', 'TOTAL']
Length: 3, dtype: str

Tipos de índice (unit):
<StringArray>
['I10_A_AVG', 'I15_A_AVG', 'RCH_A_AVG']
Length: 3, dtype: str

Combinaciones y cuántas filas tiene cada una:
purchase  unit     
DW_EXST   I10_A_AVG    36
          I15_A_AVG    37
          RCH_A_AVG    37
DW_NEW    I10_A_AVG    36
          I15_A_AVG    36
          RCH_A_AVG    36
TOTAL     I10_A_AVG    37
          I15_A_AVG    37
          RCH_A_AVG    38
dtype: int64


### Paso 1: Filtrar vivienda para quedarnos con una sola categoría

De las 9 combinaciones posibles por país/año, nos quedamos con:
- `TOTAL` (todo tipo de vivienda: nueva + segunda mano)
- `I10_A_AVG` (índice con base 2010 = 100, para ver cuánto han subido los precios)

Después de filtrar, eliminamos las columnas auxiliares (`freq`, `purchase`, `unit`)
que ya no necesitamos, y repetimos el `melt()` para pasar de formato ancho a largo.

In [27]:
# ============================================================
# FILTRAR VIVIENDA: quedarnos solo con TOTAL + I10_A_AVG
# ============================================================

# Filtramos: solo las filas donde purchase es TOTAL Y unit es I10_A_AVG
# El operador & significa "las dos condiciones a la vez"
# Los paréntesis son obligatorios cuando combinas condiciones en Pandas
df_viv_filtrado = df_vivienda[
    (df_vivienda['purchase'] == 'TOTAL') & 
    (df_vivienda['unit'] == 'I10_A_AVG')
].copy()
# .copy() crea una copia independiente para no modificar el original

# Verificamos: ahora debería haber ~37 filas (una por país)
print(f"Filas después de filtrar: {len(df_viv_filtrado)}")
print(f"Países únicos: {df_viv_filtrado['geo'].nunique()}")
print(f"\nPrimeras filas:")
print(df_viv_filtrado[['geo', '2016 ', '2017 ', '2018 ']].head())

Filas después de filtrar: 37
Países únicos: 37

Primeras filas:
    geo    2016     2017     2018 
218  AT  138.00   145.02   153.68 
219  BE  111.72   115.67   119.16 
220  BG  101.15   109.91   117.16 
221  CY   88.42    92.55    91.30 
222  CZ  112.50   125.80   136.50 


In [28]:
# ============================================================
# MELT DE VIVIENDA FILTRADA: de formato ancho a largo
# ============================================================
# Ahora que tenemos 1 fila por país, hacemos el melt igual que antes.
#
# Recordatorio de qué hace melt():
# Formato ANCHO:  geo | 2016 | 2017 | 2018  (años como columnas)
# Formato LARGO:  geo | anio | valor         (años como filas)
#
# id_vars = columnas que se quedan fijas (geo)
# Las demás columnas (los años) se "derriten" en dos columnas: anio y valor
# ============================================================

# Primero eliminamos las columnas auxiliares que ya no necesitamos
# axis=1 significa "eliminar columnas" (axis=0 sería filas)
columnas_auxiliares = ['freq', 'purchase', 'unit', df_vivienda.columns[0]]
df_viv_filtrado = df_viv_filtrado.drop(columns=columnas_auxiliares, errors='ignore')

# Ahora hacemos el melt: las columnas que NO son 'geo' son los años
df_viv_long = pd.melt(
    df_viv_filtrado,
    id_vars=['geo'],           # esta columna se queda fija
    var_name='anio',           # las columnas de años se convierten en filas
    value_name='indice_vivienda'  # los valores se llaman así
)

# Limpiamos: el año viene con espacios (ej: "2016 ") y el valor puede tener letras
df_viv_long['anio'] = df_viv_long['anio'].str.strip().astype(int)
df_viv_long['indice_vivienda'] = pd.to_numeric(df_viv_long['indice_vivienda'], errors='coerce')

# Verificamos
print(f"Vivienda limpia: {len(df_viv_long)} filas")
print(f"Filas por país/año (debería ser todo 1):")
print(df_viv_long.groupby(['geo', 'anio']).size().value_counts())
print(f"\nPrimeras filas:")
print(df_viv_long.head(10))

Vivienda limpia: 370 filas
Filas por país/año (debería ser todo 1):
1    370
Name: count, dtype: int64

Primeras filas:
    geo  anio  indice_vivienda
0    AT  2015           129.33
1    BE  2015           109.17
2    BG  2015              NaN
3    CY  2015              NaN
4    CZ  2015           105.10
5    DE  2015           119.30
6    DK  2015              NaN
7    EA  2015            98.93
8  EA19  2015            99.10
9  EA20  2015            99.05


### Paso 2: Filtrar Gini para quedarnos solo con la población total

El dataset de Gini tiene 2 filas por país/año:
- `TOTAL` = desigualdad de toda la población (el estándar)
- `Y_LT18` = desigualdad solo entre menores de 18 años

Nos quedamos con `TOTAL` y repetimos el proceso.

In [29]:
# ============================================================
# FILTRAR GINI: quedarnos solo con TOTAL (toda la población)
# ============================================================

# Paso 1: extraer las partes de la columna compuesta, igual que hicimos con vivienda
# Formato: "A,TOTAL,GINI_HND,AT" = freq,age,statinfo,geo
col_compuesta_gini = df_gini.columns[0]

partes_gini = df_gini[col_compuesta_gini].str.split(',', expand=True)
df_gini['freq'] = partes_gini[0]        # siempre "A" (anual)
df_gini['age'] = partes_gini[1]          # TOTAL o Y_LT18
df_gini['statinfo'] = partes_gini[2]     # siempre GINI_HND
df_gini['geo'] = partes_gini[3]          # código de país

# Paso 2: filtrar solo las filas de TOTAL
df_gin_filtrado = df_gini[df_gini['age'] == 'TOTAL'].copy()

print(f"Gini antes de filtrar: {len(df_gini)} filas")
print(f"Gini después de filtrar: {len(df_gin_filtrado)} filas")
print(f"Grupos de edad en el filtrado: {df_gin_filtrado['age'].unique()}")

Gini antes de filtrar: 78 filas
Gini después de filtrar: 39 filas
Grupos de edad en el filtrado: <StringArray>
['TOTAL']
Length: 1, dtype: str


In [30]:
# ============================================================
# MELT DE GINI FILTRADO
# ============================================================

# Eliminamos columnas auxiliares
columnas_aux_gini = ['freq', 'age', 'statinfo', col_compuesta_gini]
df_gin_filtrado = df_gin_filtrado.drop(columns=columnas_aux_gini, errors='ignore')

# Melt: de ancho a largo
df_gin_long = pd.melt(
    df_gin_filtrado,
    id_vars=['geo'],
    var_name='anio',
    value_name='gini'
)

# Limpieza de tipos
df_gin_long['anio'] = df_gin_long['anio'].str.strip().astype(int)
df_gin_long['gini'] = pd.to_numeric(df_gin_long['gini'], errors='coerce')

# Verificamos: ahora debería ser todo 1 fila por país/año
print(f"Gini limpio: {len(df_gin_long)} filas")
print(f"Filas por país/año (debería ser todo 1):")
print(df_gin_long.groupby(['geo', 'anio']).size().value_counts())
print(f"\nPrimeras filas:")
print(df_gin_long.head(10))

Gini limpio: 390 filas
Filas por país/año (debería ser todo 1):
1    390
Name: count, dtype: int64

Primeras filas:
    geo  anio  gini
0    AL  2016   NaN
1    AT  2016  27.2
2    BE  2016  26.3
3    BG  2016   NaN
4    CH  2016  29.4
5    CY  2016  32.1
6    CZ  2016  25.1
7    DE  2016  29.5
8    DK  2016  27.7
9  EA20  2016  30.7


### Paso 3: Repetir el merge

Ahora que vivienda tiene 1 fila por país/año y Gini también,
el merge ya no generará producto cartesiano.

Usamos `outer join` para no perder ningún dato: si un país tiene dato de PIB
pero no de Gini, la fila aparece con NaN en Gini (en vez de desaparecer).

In [31]:
# ============================================================
# MERGE DEFINITIVO
# ============================================================
# Unimos los 5 datasets paso a paso, siempre por geo + anio.
# outer join = conservar todas las filas aunque no haya coincidencia
# (las que no coinciden se rellenan con NaN)
# ============================================================

# Empezamos con salarios como base
df_merged = df_sal_anual.merge(df_pib_long, on=['geo', 'anio'], how='outer')

# Añadimos vivienda (ahora con 1 fila por país/año)
df_merged = df_merged.merge(df_viv_long, on=['geo', 'anio'], how='outer')

# Añadimos Gini (ahora con 1 fila por país/año)
df_merged = df_merged.merge(df_gin_long, on=['geo', 'anio'], how='outer')

# Añadimos brecha de género
df_merged = df_merged.merge(df_gen_long, on=['geo', 'anio'], how='outer')

print(f"DataFrame antes de filtrar:")
print(f"  Filas: {len(df_merged)} | Columnas: {len(df_merged.columns)}")
print(f"  Columnas: {df_merged.columns.tolist()}")

# ============================================================
# FILTRAR: solo países EU27 y años 2016-2024
# ============================================================

eu27 = ['AT','BE','BG','CY','CZ','DE','DK','EE','EL','ES',
        'FI','FR','HR','HU','IE','IT','LT','LU','LV','MT',
        'NL','PL','PT','RO','SE','SI','SK']

df_europa = df_merged[
    (df_merged['geo'].isin(eu27)) & 
    (df_merged['anio'].between(2016, 2024))
].copy()

print(f"\nDataFrame final (EU27, 2016-2024):")
print(f"  Filas: {len(df_europa)} | Columnas: {len(df_europa.columns)}")

# Verificación clave: ¿hay duplicados de país/año?
dupes = df_europa.groupby(['geo', 'anio']).size()
print(f"\n¿Máximo de filas por país/año?: {dupes.max()}")
print(f"(Si dice 1, el merge está bien. Si dice más de 1, sigue habiendo duplicados)")

print(f"\nPrimeras filas:")
print(df_europa.head(10))

DataFrame antes de filtrar:
  Filas: 855 | Columnas: 7
  Columnas: ['geo', 'anio', 'salario_minimo_eur', 'pib_pps', 'indice_vivienda', 'gini', 'brecha_genero']

DataFrame final (EU27, 2016-2024):
  Filas: 243 | Columnas: 7

¿Máximo de filas por país/año?: 1
(Si dice 1, el merge está bien. Si dice más de 1, sigue habiendo duplicados)

Primeras filas:
   geo  anio  salario_minimo_eur  pib_pps  indice_vivienda  gini  \
24  AT  2016                 NaN    128.0           138.00  27.2   
25  AT  2017                 NaN    125.0           145.02  27.9   
26  AT  2018                 NaN    126.0           153.68  26.8   
27  AT  2019                 NaN    124.0           162.91  27.5   
28  AT  2020                 NaN    123.0           175.25  27.0   
29  AT  2021                 NaN    121.0           195.30  26.7   
30  AT  2022                 NaN    123.0           217.90  27.8   
31  AT  2023                 NaN    121.0           211.68  28.1   
32  AT  2024                 NaN    

## Estado del merge

El merge funciona correctamente: **1 fila por país y año**, sin productos cartesianos.

Resultado actual: **243 filas y 7 columnas** (27 países EU27 × 9 años, 2016-2024).

| Requisito | Actual | Objetivo | Estado |
|-----------|--------|----------|--------|
| Filas | 243 | 1000+ | Pendiente |
| Columnas | 7 | 10+ | Pendiente |

### ¿Cómo llegamos al objetivo?

**Más columnas** (necesitamos +3):
- Salario mínimo en PPS (paridad de poder adquisitivo) → API de Eurostat
- Salario mínimo como % del salario mediano → API de Eurostat
- Potencialmente: inflación HICP → API del BCE

**Más filas** (necesitamos +757):
- Ampliar el rango de años desde 1999 (en vez de solo 2016) → API de Eurostat
- 27 países × 25 años = hasta 675 filas adicionales

### Siguiente paso
Conectar con la API de Eurostat para descargar la serie completa del salario mínimo
(indicador `earn_mw_cur`) desde 1999, con todas las unidades disponibles (EUR, PPS, etc.).

In [32]:
# ============================================================
# GUARDAMOS EL DATAFRAME ACTUAL
# ============================================================
# Guardamos el merge actual como CSV para poder cargarlo en el
# siguiente notebook sin tener que repetir toda la limpieza.
#
# index=False evita que Pandas añada una columna extra con números
# de fila que no necesitamos.
# ============================================================

df_europa.to_csv('./data/merge_parcial_eu27.csv', index=False)
print("Guardado en ./data/merge_parcial_eu27.csv")
print(f"  {len(df_europa)} filas × {len(df_europa.columns)} columnas")

Guardado en ./data/merge_parcial_eu27.csv
  243 filas × 7 columnas


# Ahora vienen las APIs (que no me gustan especialmente después de la experiencia intentando conectar con ellas en varios ejericios del Bootcamp)

## Conexión con la API de Eurostat

El dataset de salario mínimo descargado manualmente (`salario_minimo.tsv`) solo cubre
periodos recientes y una única unidad monetaria (EUR). Para el análisis necesito:

- **Serie histórica desde 1999**: el salario mínimo europeo armonizado existe desde entonces,
  y necesito esa profundidad temporal para analizar la evolución generacional (H5)
- **Salario en PPS (Paridad de Poder Adquisitivo)**: el euro nominal no refleja el coste de vida
  real de cada país. 1000€ en Luxemburgo no compran lo mismo que en Bulgaria. El ajuste PPS
  corrige esa distorsión y es fundamental para la H1

El indicador de Eurostat es `earn_mw_cur` (Minimum Wages, Current prices).
La API es pública, gratuita y no requiere autenticación.

In [37]:
# Librerías necesarias para conectar con la API
# - requests: permite hacer peticiones HTTP a servidores web
# - time / random: para añadir pausas aleatorias entre peticiones
#   y evitar que el servidor nos bloquee por exceso de solicitudes (error 429)
# - json: para interpretar la respuesta de la API, que viene en formato JSON

import requests
import time
import random
import json

print("Librerías cargadas")

Librerías cargadas


In [38]:
# Petición a la API de Eurostat
# El endpoint base es fijo; el indicador earn_mw_cur se añade al final de la URL
# sinceTimePeriod filtra desde qué periodo queremos los datos

url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/earn_mw_cur"

params = {
    'sinceTimePeriod': '1999-S1',
    'lang': 'en'
}

print("Solicitando datos a Eurostat...")
response = requests.get(url, params=params)
print(f"Código de respuesta: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"Dimensiones disponibles: {data.get('id', [])}")
    print(f"Número total de valores: {len(data.get('value', {}))}")
else:
    print(f"Error en la petición: {response.status_code}")
    print(response.text[:500])

Solicitando datos a Eurostat...
Código de respuesta: 200
Dimensiones disponibles: ['freq', 'currency', 'geo', 'time']
Número total de valores: 4484


In [39]:
# Inspección del JSON: qué dimensiones contiene y qué categorías tiene cada una
# Equivalente a hacer .head() o .unique() en un DataFrame,
# pero adaptado al formato JSON-stat que usa la API de Eurostat

for dim_name in data['id']:
    dim = data['dimension'][dim_name]
    categories = list(dim['category']['index'].keys())
    print(f"\n{dim_name}: {len(categories)} categorías")
    print(f"  Ejemplos: {categories[:15]}")
    if len(categories) > 15:
        print(f"  ... y {len(categories) - 15} más")


freq: 1 categorías
  Ejemplos: ['S']

currency: 3 categorías
  Ejemplos: ['EUR', 'PPS', 'NAC']

geo: 39 categorías
  Ejemplos: ['BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV', 'LT']
  ... y 24 más

time: 55 categorías
  Ejemplos: ['1999-S1', '1999-S2', '2000-S1', '2000-S2', '2001-S1', '2001-S2', '2002-S1', '2002-S2', '2003-S1', '2003-S2', '2004-S1', '2004-S2', '2005-S1', '2005-S2', '2006-S1']
  ... y 40 más


In [40]:
# Conversión del JSON-stat a DataFrame
# 
# El formato JSON-stat almacena los valores en una lista plana indexada
# numéricamente. Para reconstruir a qué combinación de dimensiones corresponde
# cada valor, se calcula la posición dentro de cada dimensión usando
# división entera (//) y módulo (%).
#
# Ejemplo simplificado con 2 monedas × 3 países × 2 periodos = 12 valores:
# Valor en posición 7 -> moneda 7//(3*2)=1, país (7%6)//2=0, periodo 7%2=1
# Es decir: moneda[1]=PPS, país[0]=AT, periodo[1]=S2

import numpy as np

dimensions = data['id']
dim_categories = {}
dim_sizes = []

for dim_name in dimensions:
    dim = data['dimension'][dim_name]
    cat_index = dim['category']['index']
    ordered_cats = sorted(cat_index.keys(), key=lambda x: cat_index[x])
    dim_categories[dim_name] = ordered_cats
    dim_sizes.append(len(ordered_cats))

# Multiplicadores para calcular la posición de cada dimensión
multipliers = []
for i in range(len(dim_sizes)):
    mult = 1
    for j in range(i + 1, len(dim_sizes)):
        mult *= dim_sizes[j]
    multipliers.append(mult)

# Reconstrucción del DataFrame fila por fila
values = data['value']
rows = []

for str_idx, val in values.items():
    idx = int(str_idx)
    row = {}
    for i, dim_name in enumerate(dimensions):
        cat_position = (idx // multipliers[i]) % dim_sizes[i]
        row[dim_name] = dim_categories[dim_name][cat_position]
    row['valor'] = val
    rows.append(row)

df_api = pd.DataFrame(rows)

print(f"DataFrame de la API: {len(df_api)} filas × {len(df_api.columns)} columnas")
print(f"Columnas: {df_api.columns.tolist()}")
print(df_api.head(10))

DataFrame de la API: 4484 filas × 5 columnas
Columnas: ['freq', 'currency', 'geo', 'time', 'valor']
  freq currency geo     time  valor
0    S      EUR  AL  1999-S2   45.0
1    S      EUR  AL  2000-S1   47.0
2    S      EUR  AL  2000-S2   52.0
3    S      EUR  AL  2001-S1   53.0
4    S      EUR  AL  2001-S2   60.0
5    S      EUR  AL  2002-S1   63.0
6    S      EUR  AL  2002-S2   68.0
7    S      EUR  AL  2003-S1   67.0
8    S      EUR  AL  2003-S2   74.0
9    S      EUR  AL  2004-S1   75.0


In [41]:
# Inspección del DataFrame resultante de la API

for col in df_api.columns:
    if col != 'valor':
        vals = df_api[col].unique()
        print(f"\n{col}: {len(vals)} valores únicos")
        print(f"  {list(vals[:20])}")

print(f"\nRango de valores numéricos: {df_api['valor'].min():.2f} - {df_api['valor'].max():.2f}")
print(f"Nulos en valor: {df_api['valor'].isna().sum()}")


freq: 1 valores únicos
  ['S']

currency: 3 valores únicos
  ['EUR', 'NAC', 'PPS']

geo: 31 valores únicos
  ['AL', 'BE', 'BG', 'CY', 'CZ', 'DE', 'EE', 'EL', 'ES', 'FR', 'HR', 'HU', 'IE', 'LT', 'LU', 'LV', 'MD', 'ME', 'MK', 'MT']

time: 55 valores únicos
  ['1999-S2', '2000-S1', '2000-S2', '2001-S1', '2001-S2', '2002-S1', '2002-S2', '2003-S1', '2003-S2', '2004-S1', '2004-S2', '2005-S1', '2005-S2', '2006-S1', '2006-S2', '2007-S1', '2007-S2', '2008-S1', '2008-S2', '2009-S1']

Rango de valores numéricos: 2.00 - 322800.00
Nulos en valor: 0


In [42]:
# Filtrado por unidades de interés
# EUR = euros nominales (el número "bruto" que aparece en el BOE de cada país)
# PPS = Purchasing Power Standard (paridad de poder adquisitivo)
# Se descarta NAC (moneda nacional) porque no permite comparar entre países

df_api_filtrado = df_api[df_api['currency'].isin(['EUR', 'PPS'])].copy()

# Extraer año y semestre del campo time
# El formato es "2020-S1", así que los primeros 4 caracteres son el año
# y los últimos 2 son el semestre (S1 o S2)
df_api_filtrado['anio'] = df_api_filtrado['time'].str[:4].astype(int)
df_api_filtrado['semestre'] = df_api_filtrado['time'].str[-2:]

print(f"Filas tras filtrar EUR y PPS: {len(df_api_filtrado)}")
print(f"Rango temporal: {df_api_filtrado['anio'].min()} - {df_api_filtrado['anio'].max()}")
print(f"Semestres: {df_api_filtrado['semestre'].unique()}")
print(df_api_filtrado[['geo', 'anio', 'semestre', 'currency', 'valor']].head(10))

Filas tras filtrar EUR y PPS: 2950
Rango temporal: 1999 - 2026
Semestres: <StringArray>
['S2', 'S1']
Length: 2, dtype: str
  geo  anio semestre currency  valor
0  AL  1999       S2      EUR   45.0
1  AL  2000       S1      EUR   47.0
2  AL  2000       S2      EUR   52.0
3  AL  2001       S1      EUR   53.0
4  AL  2001       S2      EUR   60.0
5  AL  2002       S1      EUR   63.0
6  AL  2002       S2      EUR   68.0
7  AL  2003       S1      EUR   67.0
8  AL  2003       S2      EUR   74.0
9  AL  2004       S1      EUR   75.0


In [43]:
# Pivotado: convertir las filas de EUR y PPS en columnas separadas
# Antes del pivot, cada combinación país/año/semestre tiene DOS filas:
#   ES | 2020 | S1 | EUR | 1108.0
#   ES | 2020 | S1 | PPS | 1050.0
#
# Después del pivot, tiene UNA fila con DOS columnas:
#   ES | 2020 | S1 | salario_min_eur=1108.0 | salario_min_pps=1050.0
#
# pivot_table() es la operación inversa a melt():
#   melt()        -> convierte columnas en filas (formato ancho a largo)
#   pivot_table() -> convierte filas en columnas (formato largo a ancho)

df_pivot = df_api_filtrado.pivot_table(
    index=['geo', 'anio', 'semestre'],  # columnas que se quedan como filas
    columns='currency',                  # los valores de currency se convierten en columnas
    values='valor',                      # los números que rellenan las nuevas columnas
    aggfunc='first'                      # si hay duplicados, se conserva el primero
).reset_index()

# Limpiar el nombre del grupo de columnas que deja pivot_table
df_pivot.columns.name = None

# Renombrar para mayor claridad
df_pivot = df_pivot.rename(columns={
    'EUR': 'salario_min_eur',
    'PPS': 'salario_min_pps'
})

print(f"Tras pivotar: {len(df_pivot)} filas × {len(df_pivot.columns)} columnas")
print(f"Columnas: {df_pivot.columns.tolist()}")
print(df_pivot.head(10))

Tras pivotar: 1532 filas × 5 columnas
Columnas: ['geo', 'anio', 'semestre', 'salario_min_eur', 'salario_min_pps']
  geo  anio semestre  salario_min_eur  salario_min_pps
0  AL  1999       S1              NaN             85.0
1  AL  1999       S2             45.0             93.0
2  AL  2000       S1             47.0             94.0
3  AL  2000       S2             52.0            104.0
4  AL  2001       S1             53.0            103.0
5  AL  2001       S2             60.0            112.0
6  AL  2002       S1             63.0            110.0
7  AL  2002       S2             68.0            137.0
8  AL  2003       S1             67.0            138.0
9  AL  2003       S2             74.0            148.0


In [44]:
# Media anual: promedio de S1 (enero) y S2 (julio) para obtener un valor por país y año
# Mismo proceso que aplicamos al dataset de salarios del TSV
#
# groupby(['geo', 'anio']) agrupa las filas que comparten país y año
# .mean(numeric_only=True) calcula la media solo de columnas numéricas
#   (numeric_only=True evita que intente calcular la media de 'semestre', que es texto)
# .reset_index() devuelve el resultado a un DataFrame normal

df_salarios_api = df_pivot.groupby(['geo', 'anio']).mean(numeric_only=True).reset_index()

# Filtrar solo los 27 países de la UE (descartar AL, MD, ME, MK, etc.)
df_salarios_api = df_salarios_api[df_salarios_api['geo'].isin(eu27)].copy()

print(f"Salarios API (EU27, media anual): {len(df_salarios_api)} filas")
print(f"Columnas: {df_salarios_api.columns.tolist()}")
print(f"Rango: {df_salarios_api['anio'].min()} - {df_salarios_api['anio'].max()}")
print(f"Países: {df_salarios_api['geo'].nunique()}")
print(f"\nPrimeras filas:")
print(df_salarios_api.head(10))
print(f"\nÚltimas filas:")
print(df_salarios_api.tail(10))

Salarios API (EU27, media anual): 566 filas
Columnas: ['geo', 'anio', 'salario_min_eur', 'salario_min_pps']
Rango: 1999 - 2026
Países: 22

Primeras filas:
   geo  anio  salario_min_eur  salario_min_pps
28  BE  1999           1085.0            960.0
29  BE  2000           1096.0           1005.0
30  BE  2001           1129.0           1031.0
31  BE  2002           1151.5           1064.5
32  BE  2003           1174.5           1072.5
33  BE  2004           1186.0           1083.0
34  BE  2005           1210.0           1108.0
35  BE  2006           1234.0           1124.0
36  BE  2007           1271.0           1152.0
37  BE  2008           1323.0           1194.5

Últimas filas:
    geo  anio  salario_min_eur  salario_min_pps
668  SK  2017            435.0            528.0
669  SK  2018            480.0            568.0
670  SK  2019            520.0            604.0
671  SK  2020            580.0            697.0
672  SK  2021            623.0            756.0
673  SK  2022           

## Merge definitivo

Se rehace el merge utilizando los datos de salario de la API como base, ya que aportan:
- Serie histórica desde 1999 (vs. solo periodos recientes del TSV descargado)
- Dos columnas de salario: EUR (nominal) y PPS (ajustado a poder adquisitivo)

Se mantiene el outer join para conservar el máximo de datos posible.
Las combinaciones de país/año donde solo existe dato de salario (años 1999-2005, por ejemplo)
pero no de PIB o vivienda, aparecerán con NaN en esas columnas.

In [45]:
# Merge definitivo: salarios API + PIB + vivienda + Gini + brecha de género
# Se usa outer join para conservar todas las filas aunque algún dataset
# no tenga dato para esa combinación de país/año

df_final = df_salarios_api.merge(df_pib_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_viv_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gin_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gen_long, on=['geo', 'anio'], how='outer')

# Filtrar EU27 (el outer join puede haber traído países de otros datasets)
df_final = df_final[df_final['geo'].isin(eu27)].copy()

# Ordenar para legibilidad
df_final = df_final.sort_values(['geo', 'anio']).reset_index(drop=True)

print("=" * 60)
print("MERGE DEFINITIVO")
print("=" * 60)
print(f"Filas: {len(df_final)}")
print(f"Columnas: {len(df_final.columns)}")
print(f"Nombres: {df_final.columns.tolist()}")
print(f"Rango temporal: {df_final['anio'].min()} - {df_final['anio'].max()}")
print(f"Países: {df_final['geo'].nunique()}")

dupes = df_final.groupby(['geo', 'anio']).size()
print(f"\nMáx filas por país/año: {dupes.max()} (debe ser 1)")

print(f"\nNulos por columna:")
print(df_final.isnull().sum())

print(f"\n% de nulos por columna:")
print((df_final.isnull().sum() / len(df_final) * 100).round(1))

print(f"\nPrimeras filas:")
print(df_final.head(15))

MERGE DEFINITIVO
Filas: 693
Columnas: 8
Nombres: ['geo', 'anio', 'salario_min_eur', 'salario_min_pps', 'pib_pps', 'indice_vivienda', 'gini', 'brecha_genero']
Rango temporal: 1999 - 2026
Países: 27

Máx filas por país/año: 1 (debe ser 1)

Nulos por columna:
geo                  0
anio                 0
salario_min_eur    127
salario_min_pps    127
pib_pps            384
indice_vivienda    440
gini               453
brecha_genero      260
dtype: int64

% de nulos por columna:
geo                 0.0
anio                0.0
salario_min_eur    18.3
salario_min_pps    18.3
pib_pps            55.4
indice_vivienda    63.5
gini               65.4
brecha_genero      37.5
dtype: float64

Primeras filas:
   geo  anio  salario_min_eur  salario_min_pps  pib_pps  indice_vivienda  \
0   AT  2006              NaN              NaN      NaN              NaN   
1   AT  2007              NaN              NaN      NaN              NaN   
2   AT  2008              NaN              NaN      NaN              

In [47]:
# Guardar dataset definitivo

df_final.to_csv('./data/dataset_europa_final.csv', index=False)

print(f"Guardado: ./data/dataset_europa_final.csv")
print(f"  {len(df_final)} filas × {len(df_final.columns)} columnas")
print(f"\nVERIFICACIÓN DE REQUISITOS:")
print(f"  Filas:    {len(df_final)} ")
print(f"  Columnas: {len(df_final.columns)}")

Guardado: ./data/dataset_europa_final.csv
  693 filas × 8 columnas

VERIFICACIÓN DE REQUISITOS:
  Filas:    693 
  Columnas: 8


## Datasets adicionales de Eurostat

Con los 5 datasets iniciales obtengo 693 filas y 8 columnas. Para cumplir los requisitos
mínimos del proyecto (1000+ filas, 10+ columnas) y fortalecer las hipótesis, descargo
3 indicadores adicionales de la API de Eurostat:

| Indicador | Código Eurostat | Columna nueva | Hipótesis relacionada |
|-----------|----------------|---------------|----------------------|
| Tasa de pobreza laboral | `ilc_iw01` | `pobreza_laboral` | H4: el empleo no previene la pobreza |
| Tasa de desempleo | `une_rt_a` | `tasa_desempleo` | H4 y H5: contexto del mercado laboral |
| Inflación (HICP) | `prc_hicp_aind` | `inflacion_hicp` | H5: salario real ajustado por inflación |

Los tres se descargan de la API pública de Eurostat con el mismo método utilizado
para el salario mínimo. Se aplica `time.sleep()` entre peticiones para evitar
sobrecarga del servidor.

In [48]:
# ============================================================
# FUNCIÓN REUTILIZABLE PARA DESCARGAR DE LA API DE EUROSTAT
# ============================================================
# En vez de repetir el mismo bloque de código 3 veces (una por cada
# indicador nuevo), creo una función que encapsula todo el proceso:
# petición HTTP -> parseo del JSON-stat -> conversión a DataFrame.
#
# Una función es un bloque de código con nombre que puedo llamar
# cuando quiera pasándole parámetros diferentes. Es como crear
# mi propia herramienta dentro de Python.
#
# def nombre(parametros):  -> define la función
#     ...código...
#     return resultado     -> devuelve el resultado
#
# Luego la llamo así: resultado = nombre(parametros)
# ============================================================

def descargar_eurostat(indicador, desde='1999-S1'):
    """
    Descarga un indicador de la API de Eurostat y lo devuelve como DataFrame.
    
    Parámetros:
    - indicador: código del dataset en Eurostat (ej: 'ilc_iw01')
    - desde: periodo inicial (por defecto '1999-S1')
    
    Devuelve: DataFrame con columnas por cada dimensión + columna 'valor'
    """
    
    url = f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/{indicador}"
    params = {'sinceTimePeriod': desde, 'lang': 'en'}
    
    print(f"Descargando {indicador}...")
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"  ERROR: código {response.status_code}")
        return None
    
    data = response.json()
    dimensions = data['id']
    
    # Extraer categorías ordenadas de cada dimensión
    dim_categories = {}
    dim_sizes = []
    for dim_name in dimensions:
        dim = data['dimension'][dim_name]
        cat_index = dim['category']['index']
        ordered_cats = sorted(cat_index.keys(), key=lambda x: cat_index[x])
        dim_categories[dim_name] = ordered_cats
        dim_sizes.append(len(ordered_cats))
    
    # Calcular multiplicadores para decodificar los índices
    multipliers = []
    for i in range(len(dim_sizes)):
        mult = 1
        for j in range(i + 1, len(dim_sizes)):
            mult *= dim_sizes[j]
        multipliers.append(mult)
    
    # Reconstruir cada fila a partir del índice plano
    values = data['value']
    rows = []
    for str_idx, val in values.items():
        idx = int(str_idx)
        row = {}
        for i, dim_name in enumerate(dimensions):
            cat_position = (idx // multipliers[i]) % dim_sizes[i]
            row[dim_name] = dim_categories[dim_name][cat_position]
        row['valor'] = val
        rows.append(row)
    
    df = pd.DataFrame(rows)
    print(f"  OK: {len(df)} filas × {len(df.columns)} columnas")
    print(f"  Dimensiones: {dimensions}")
    
    return df

print("Función descargar_eurostat() definida")

Función descargar_eurostat() definida


In [49]:
# ============================================================
# DESCARGA DE LOS 3 INDICADORES ADICIONALES
# ============================================================
# Entre cada petición añado una pausa aleatoria de 1 a 3 segundos
# para no sobrecargar el servidor de Eurostat.
# random.uniform(1, 3) genera un número decimal aleatorio entre 1 y 3.
# time.sleep() pausa la ejecución ese número de segundos.
# ============================================================

# 1. Tasa de pobreza laboral (ilc_iw01)
#    Porcentaje de personas empleadas que están en riesgo de pobreza.
#    Directamente vinculado a la H4: "tener trabajo no te saca de pobre"
df_pobreza_raw = descargar_eurostat('ilc_iw01')

time.sleep(random.uniform(1, 3))  # pausa para no saturar el servidor

# 2. Tasa de desempleo anual (une_rt_a)
#    Porcentaje de población activa que no tiene empleo.
#    Contexto necesario para H4 y H5
df_desempleo_raw = descargar_eurostat('une_rt_a')

time.sleep(random.uniform(1, 3))

# 3. Inflación HICP - índice anual (prc_hicp_aind)
#    Índice Armonizado de Precios al Consumo.
#    Necesario para la H5: calcular si el salario mínimo real (descontando
#    la inflación) ha subido o bajado para la generación Z
df_inflacion_raw = descargar_eurostat('prc_hicp_aind')

print("\n" + "=" * 50)
print("Las 3 descargas se completaron")

Descargando ilc_iw01...
  OK: 64316 filas × 8 columnas
  Dimensiones: ['freq', 'wstatus', 'sex', 'age', 'unit', 'geo', 'time']
Descargando une_rt_a...
  OK: 39669 filas × 7 columnas
  Dimensiones: ['freq', 'age', 'unit', 'sex', 'geo', 'time']
Descargando prc_hicp_aind...
  ERROR: código 413

Las 3 descargas se completaron


In [50]:
# ============================================================
# INSPECCIÓN DE LOS 3 DATASETS DESCARGADOS
# ============================================================
# Antes de limpiar, siempre miro qué he recibido.
# Necesito saber: qué columnas tiene cada uno, qué valores únicos
# hay en cada dimensión, y cómo se llaman las cosas.
# ============================================================

datasets_raw = {
    'Pobreza laboral': df_pobreza_raw,
    'Desempleo': df_desempleo_raw,
    'Inflación HICP': df_inflacion_raw
}

for nombre, df in datasets_raw.items():
    print(f"\n{'=' * 50}")
    print(f"{nombre}")
    print(f"{'=' * 50}")
    print(f"Filas: {len(df)} | Columnas: {df.columns.tolist()}")
    for col in df.columns:
        if col != 'valor':
            vals = df[col].unique()
            print(f"  {col}: {len(vals)} únicos -> {list(vals[:10])}")


Pobreza laboral
Filas: 64316 | Columnas: ['freq', 'wstatus', 'sex', 'age', 'unit', 'geo', 'time', 'valor']
  freq: 1 únicos -> ['A']
  wstatus: 3 únicos -> ['EMP', 'NSAL', 'SAL']
  sex: 3 únicos -> ['F', 'M', 'T']
  age: 12 únicos -> ['Y16-19', 'Y16-24', 'Y16-29', 'Y18-24', 'Y18-64', 'Y20-24', 'Y20-29', 'Y25-29', 'Y25-54', 'Y55-64']
  unit: 1 únicos -> ['PC']
  geo: 45 únicos -> ['AL', 'AT', 'CH', 'CY', 'DE', 'DK', 'EA18', 'EA19', 'EE', 'EL']
  time: 23 únicos -> ['2017', '2018', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010']

Desempleo
Filas: 39669 | Columnas: ['freq', 'age', 'unit', 'sex', 'geo', 'time', 'valor']
  freq: 1 únicos -> ['A']
  age: 7 únicos -> ['Y15-24', 'Y15-29', 'Y15-74', 'Y20-64', 'Y25-54', 'Y25-74', 'Y55-74']
  unit: 3 únicos -> ['PC_ACT', 'PC_POP', 'THS_PER']
  sex: 3 únicos -> ['F', 'M', 'T']
  geo: 38 únicos -> ['AT', 'BA', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EA20']
  time: 23 únicos -> ['2009', '2010', '2011', '2012', '2013', '2014', '20

TypeError: object of type 'NoneType' has no len()

### Descarga de inflación: petición filtrada

El dataset de inflación HICP (`prc_hicp_aind`) es demasiado grande para descargarlo
completo (error 413). Añado filtros directamente en la petición para pedir
solo lo que necesito: índice general de precios (COICOP = CP00), índice anual (unit = INX_A_AVG),
todos los países y todos los años disponibles.

In [51]:
# ============================================================
# DESCARGA FILTRADA DE INFLACIÓN HICP
# ============================================================
# El error 413 (Request Entity Too Large) significa que el servidor
# rechaza la petición porque el dataset completo es demasiado grande.
# Solución: añadir filtros en los parámetros de la petición para pedir
# solo la porción que necesito.
#
# COICOP es la clasificación europea de productos de consumo.
# CP00 = "All items" (todos los productos), que es el índice general
# de inflación. Sin este filtro, la API intentaría devolverme datos
# para cientos de categorías (alimentación, transporte, ropa, etc.)
# ============================================================

url_hicp = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/prc_hicp_aind"

params_hicp = {
    'sinceTimePeriod': '1999',
    'coicop': 'CP00',          # índice general (todos los productos)
    'unit': 'INX_A_AVG',       # índice anual medio (base 2015 = 100)
    'lang': 'en'
}

print("Descargando inflación HICP con filtros...")
response_hicp = requests.get(url_hicp, params=params_hicp)
print(f"Código de respuesta: {response_hicp.status_code}")

if response_hicp.status_code == 200:
    data_hicp = response_hicp.json()
    
    # Uso la misma lógica de conversión que la función descargar_eurostat()
    dimensions = data_hicp['id']
    dim_categories = {}
    dim_sizes = []
    
    for dim_name in dimensions:
        dim = data_hicp['dimension'][dim_name]
        cat_index = dim['category']['index']
        ordered_cats = sorted(cat_index.keys(), key=lambda x: cat_index[x])
        dim_categories[dim_name] = ordered_cats
        dim_sizes.append(len(ordered_cats))
    
    multipliers = []
    for i in range(len(dim_sizes)):
        mult = 1
        for j in range(i + 1, len(dim_sizes)):
            mult *= dim_sizes[j]
        multipliers.append(mult)
    
    values = data_hicp['value']
    rows = []
    for str_idx, val in values.items():
        idx = int(str_idx)
        row = {}
        for i, dim_name in enumerate(dimensions):
            cat_position = (idx // multipliers[i]) % dim_sizes[i]
            row[dim_name] = dim_categories[dim_name][cat_position]
        row['valor'] = val
        rows.append(row)
    
    df_inflacion_raw = pd.DataFrame(rows)
    print(f"OK: {len(df_inflacion_raw)} filas × {len(df_inflacion_raw.columns)} columnas")
    print(f"Columnas: {df_inflacion_raw.columns.tolist()}")
    
    # Inspección rápida
    for col in df_inflacion_raw.columns:
        if col != 'valor':
            vals = df_inflacion_raw[col].unique()
            print(f"  {col}: {len(vals)} únicos -> {list(vals[:10])}")
else:
    print(f"ERROR: {response_hicp.status_code}")
    print(response_hicp.text[:500])

Descargando inflación HICP con filtros...
Código de respuesta: 200
OK: 1127 filas × 6 columnas
Columnas: ['freq', 'unit', 'coicop', 'geo', 'time', 'valor']
  freq: 1 únicos -> ['A']
  unit: 1 únicos -> ['INX_A_AVG']
  coicop: 1 únicos -> ['CP00']
  geo: 45 únicos -> ['AL', 'AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EA']
  time: 27 únicos -> ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


### Limpieza de los 3 datasets adicionales

Cada dataset tiene múltiples categorías por país y año (sexo, edad, tipo de empleo...).
Para evitar el mismo problema de producto cartesiano que tuve con vivienda y Gini,
filtro cada dataset para quedarme con **una sola fila por país y año** antes del merge:

- **Pobreza laboral**: `sex=T` (total, ambos sexos), `age=Y18-64` (población en edad laboral),
  `wstatus=EMP` (personas empleadas). Así obtengo: "de las personas empleadas de 18-64 años,
  ¿qué porcentaje está en riesgo de pobreza?"
- **Desempleo**: `sex=T` (total), `age=Y15-74` (rango estándar de Eurostat para tasa de desempleo),
  `unit=PC_ACT` (porcentaje de población activa)
- **Inflación**: ya viene filtrada de la descarga (CP00 + INX_A_AVG), solo extraigo país y año

In [52]:
# ============================================================
# LIMPIEZA DE POBREZA LABORAL (ilc_iw01)
# ============================================================
# El dataset tiene 64316 filas porque combina 3 sexos × 12 edades
# × 3 tipos de empleo × 45 países × 23 años.
# Filtro para quedarme con una sola combinación relevante:
# - wstatus = 'EMP' (todos los empleados, no solo asalariados)
# - sex = 'T' (total, ambos sexos juntos)
# - age = 'Y18-64' (población en edad laboral estándar)
# ============================================================

df_pobreza = df_pobreza_raw[
    (df_pobreza_raw['wstatus'] == 'EMP') &
    (df_pobreza_raw['sex'] == 'T') &
    (df_pobreza_raw['age'] == 'Y18-64')
].copy()

# Extraer año (viene como "2020", ya es formato año limpio)
df_pobreza['anio'] = df_pobreza['time'].astype(int)

# Renombrar geo si hace falta y quedarme solo con las columnas necesarias
df_pobreza = df_pobreza.rename(columns={'valor': 'pobreza_laboral'})
df_pobreza = df_pobreza[['geo', 'anio', 'pobreza_laboral']]

# Verificar: 1 fila por país/año
dupes = df_pobreza.groupby(['geo', 'anio']).size()
print(f"Pobreza laboral limpia: {len(df_pobreza)} filas")
print(f"Máx filas por país/año: {dupes.max()} (debe ser 1)")
print(f"Países: {df_pobreza['geo'].nunique()}")
print(f"Rango: {df_pobreza['anio'].min()} - {df_pobreza['anio'].max()}")
print(df_pobreza.head())

Pobreza laboral limpia: 815 filas
Máx filas por país/año: 1 (debe ser 1)
Países: 45
Rango: 2003 - 2025
      geo  anio  pobreza_laboral
21456  AL  2017             17.9
21457  AL  2018             16.6
21458  AL  2019             14.7
21459  AL  2020             12.8
21460  AL  2021             12.7


In [53]:
# ============================================================
# LIMPIEZA DE DESEMPLEO (une_rt_a)
# ============================================================
# Filtro con los valores estándar de Eurostat para tasa de desempleo:
# - sex = 'T' (total, ambos sexos)
# - age = 'Y15-74' (rango estándar para tasa de desempleo en la UE)
# - unit = 'PC_ACT' (porcentaje de la población activa)
#   Las otras opciones son PC_POP (% de toda la población, incluidos
#   niños y jubilados) y THS_PER (miles de personas, número absoluto).
#   PC_ACT es la medida estándar que sale en los titulares de prensa.
# ============================================================

df_desempleo = df_desempleo_raw[
    (df_desempleo_raw['sex'] == 'T') &
    (df_desempleo_raw['age'] == 'Y15-74') &
    (df_desempleo_raw['unit'] == 'PC_ACT')
].copy()

df_desempleo['anio'] = df_desempleo['time'].astype(int)
df_desempleo = df_desempleo.rename(columns={'valor': 'tasa_desempleo'})
df_desempleo = df_desempleo[['geo', 'anio', 'tasa_desempleo']]

dupes = df_desempleo.groupby(['geo', 'anio']).size()
print(f"Desempleo limpio: {len(df_desempleo)} filas")
print(f"Máx filas por país/año: {dupes.max()} (debe ser 1)")
print(f"Países: {df_desempleo['geo'].nunique()}")
print(f"Rango: {df_desempleo['anio'].min()} - {df_desempleo['anio'].max()}")
print(df_desempleo.head())

Desempleo limpio: 634 filas
Máx filas por país/año: 1 (debe ser 1)
Países: 38
Rango: 2003 - 2025
      geo  anio  tasa_desempleo
12605  AT  2009             5.7
12606  AT  2010             5.2
12607  AT  2011             4.9
12608  AT  2012             5.2
12609  AT  2013             5.7


In [54]:
# ============================================================
# LIMPIEZA DE INFLACIÓN HICP (prc_hicp_aind)
# ============================================================
# Este dataset ya viene filtrado de la descarga (CP00 + INX_A_AVG),
# así que solo necesito extraer el país, el año y el valor.
# El valor es un índice con base 2015 = 100. Si un país tiene 115
# en 2020, significa que los precios subieron un 15% desde 2015.
# ============================================================

# Identificar la columna de geo y time
print(f"Columnas de inflación: {df_inflacion_raw.columns.tolist()}")

# Identificar columna geo (puede llamarse 'geo' o 'geo\\TIME_PERIOD')
col_geo_inf = [c for c in df_inflacion_raw.columns if 'geo' in c.lower()][0]
col_time_inf = [c for c in df_inflacion_raw.columns if 'time' in c.lower()][0]

df_inflacion = df_inflacion_raw.copy()
df_inflacion = df_inflacion.rename(columns={col_geo_inf: 'geo', col_time_inf: 'time_raw'})
df_inflacion['anio'] = df_inflacion['time_raw'].astype(int)
df_inflacion = df_inflacion.rename(columns={'valor': 'inflacion_hicp'})
df_inflacion = df_inflacion[['geo', 'anio', 'inflacion_hicp']]

dupes = df_inflacion.groupby(['geo', 'anio']).size()
print(f"\nInflación limpia: {len(df_inflacion)} filas")
print(f"Máx filas por país/año: {dupes.max()} (debe ser 1)")
print(f"Países: {df_inflacion['geo'].nunique()}")
print(f"Rango: {df_inflacion['anio'].min()} - {df_inflacion['anio'].max()}")
print(df_inflacion.head())

Columnas de inflación: ['freq', 'unit', 'coicop', 'geo', 'time', 'valor']

Inflación limpia: 1127 filas
Máx filas por país/año: 1 (debe ser 1)
Países: 45
Rango: 1999 - 2025
  geo  anio  inflacion_hicp
0  AL  2016          101.51
1  AL  2017          104.76
2  AL  2018          106.59
3  AL  2019          108.39
4  AL  2020          110.74


## Merge definitivo con todos los datasets

Ahora tengo 8 datasets limpios, cada uno con exactamente 1 fila por país/año:

| Dataset | Columna(s) | Fuente |
|---------|-----------|--------|
| Salarios API | `salario_min_eur`, `salario_min_pps` | API Eurostat `earn_mw_cur` |
| PIB per cápita | `pib_pps` | TSV Eurostat `tec00114` |
| Precios vivienda | `indice_vivienda` | TSV Eurostat `prc_hpi_a` |
| Coeficiente Gini | `gini` | TSV Eurostat `ilc_di12` |
| Brecha de género | `brecha_genero` | TSV Eurostat `sdg_05_20` |
| Pobreza laboral | `pobreza_laboral` | API Eurostat `ilc_iw01` |
| Tasa de desempleo | `tasa_desempleo` | API Eurostat `une_rt_a` |
| Inflación HICP | `inflacion_hicp` | API Eurostat `prc_hicp_aind` |

El merge se hace con outer join sobre `geo + anio` para conservar el máximo de datos.
Se filtra a EU27 al final.

In [55]:
# ============================================================
# MERGE DEFINITIVO: 8 datasets sobre geo + anio
# ============================================================
# Empiezo con los salarios de la API como base y voy añadiendo
# cada dataset con outer join.
# Outer join = conservar todas las filas aunque no haya coincidencia.
# Los huecos se rellenan con NaN.
# ============================================================

df_final = df_salarios_api.merge(df_pib_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_viv_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gin_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gen_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_pobreza, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_desempleo, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_inflacion, on=['geo', 'anio'], how='outer')

# Filtrar solo EU27
df_final = df_final[df_final['geo'].isin(eu27)].copy()

# Ordenar
df_final = df_final.sort_values(['geo', 'anio']).reset_index(drop=True)

# Verificación completa
print("=" * 60)
print("MERGE DEFINITIVO")
print("=" * 60)
print(f"Filas: {len(df_final)}")
print(f"Columnas: {len(df_final.columns)}")
print(f"Nombres: {df_final.columns.tolist()}")
print(f"Rango temporal: {df_final['anio'].min()} - {df_final['anio'].max()}")
print(f"Países: {df_final['geo'].nunique()}")

dupes = df_final.groupby(['geo', 'anio']).size()
print(f"\nMáx filas por país/año: {dupes.max()} (debe ser 1)")

print(f"\nNulos por columna:")
print(df_final.isnull().sum())

print(f"\n% de nulos por columna:")
print((df_final.isnull().sum() / len(df_final) * 100).round(1))

print(f"\nPrimeras filas:")
print(df_final.head(15))

MERGE DEFINITIVO
Filas: 751
Columnas: 11
Nombres: ['geo', 'anio', 'salario_min_eur', 'salario_min_pps', 'pib_pps', 'indice_vivienda', 'gini', 'brecha_genero', 'pobreza_laboral', 'tasa_desempleo', 'inflacion_hicp']
Rango temporal: 1999 - 2026
Países: 27

Máx filas por país/año: 1 (debe ser 1)

Nulos por columna:
geo                  0
anio                 0
salario_min_eur    185
salario_min_pps    185
pib_pps            442
indice_vivienda    498
gini               511
brecha_genero      318
pobreza_laboral    181
tasa_desempleo     282
inflacion_hicp      22
dtype: int64

% de nulos por columna:
geo                 0.0
anio                0.0
salario_min_eur    24.6
salario_min_pps    24.6
pib_pps            58.9
indice_vivienda    66.3
gini               68.0
brecha_genero      42.3
pobreza_laboral    24.1
tasa_desempleo     37.5
inflacion_hicp      2.9
dtype: float64

Primeras filas:
   geo  anio  salario_min_eur  salario_min_pps  pib_pps  indice_vivienda  \
0   AT  1999            

In [56]:
# ============================================================
# GUARDAR DATASET DEFINITIVO
# ============================================================

df_final.to_csv('./data/dataset_europa_final.csv', index=False)

print(f"Guardado: ./data/dataset_europa_final.csv")
print(f"  {len(df_final)} filas × {len(df_final.columns)} columnas")

print(f"\n{'=' * 50}")
print(f"VERIFICACIÓN DE REQUISITOS")
print(f"{'=' * 50}")
print(f"  Filas:    {len(df_final)} {'✓' if len(df_final) >= 1000 else '✗ (necesito 1000+)'}")
print(f"  Columnas: {len(df_final.columns)} {'✓' if len(df_final.columns) >= 10 else '✗ (necesito 10+)'}")

Guardado: ./data/dataset_europa_final.csv
  751 filas × 11 columnas

VERIFICACIÓN DE REQUISITOS
  Filas:    751 ✗ (necesito 1000+)
  Columnas: 11 ✓


## Ampliación del análisis: de EU27 a toda Europa

Inicialmente el análisis se limitaba a los 27 países de la UE. Sin embargo, al construir
el dataset observo que los datos de Eurostat incluyen países europeos que no son miembros
de la UE: Noruega, Suiza, Islandia, Reino Unido, Turquía, Serbia y los Balcanes occidentales.

Incluirlos enriquece el análisis por varias razones:

- **Noruega, Suiza e Islandia** tienen algunos de los salarios más altos de Europa pero no
  pertenecen a la UE. ¿Su bienestar laboral depende de la pertenencia al bloque comunitario
  o de otros factores?
- **Turquía y los Balcanes** (Serbia, Montenegro, Macedonia del Norte, Albania, Bosnia)
  son candidatos o potenciales candidatos a entrar en la UE. Comparar sus condiciones
  laborales con las de los miembros actuales muestra la brecha que existe
- **Reino Unido** salió de la UE (Brexit). Tener sus datos permite observar si hay
  diferencias en la evolución post-salida

La UE27 no existe en el vacío. Comparar el bloque comunitario con sus vecinos revela si
el proyecto europeo realmente protege a sus trabajadores o si el bienestar depende de
factores que trascienden la pertenencia a la UE.

In [57]:
# ============================================================
# DEFINIR LA LISTA AMPLIADA DE PAÍSES EUROPEOS
# ============================================================
# Amplío la lista de EU27 con los países europeos adicionales que
# aparecen en los datasets de Eurostat.
#
# Para saber qué países están disponibles, reviso los valores únicos
# de la columna 'geo' en los datasets descargados. Cada código de
# país es un estándar ISO 3166-1 alpha-2 (dos letras).
#
# Excluyo los agregados (EA20 = zona euro, EU27_2020 = media UE)
# porque no son países reales, son promedios estadísticos que
# distorsionarían el análisis.
# ============================================================

# EU27: los 27 miembros actuales de la Unión Europea
eu27 = ['AT','BE','BG','CY','CZ','DE','DK','EE','EL','ES',
        'FI','FR','HR','HU','IE','IT','LT','LU','LV','MT',
        'NL','PL','PT','RO','SE','SI','SK']

# Países europeos adicionales presentes en los datos de Eurostat
extra_europeos = [
    'NO',  # Noruega (EEE, no UE)
    'CH',  # Suiza (acuerdos bilaterales con UE, no miembro)
    'IS',  # Islandia (EEE, no UE)
    'UK',  # Reino Unido (ex-UE, salió en 2020)
    'RS',  # Serbia (candidato a UE)
    'TR',  # Turquía (candidato a UE)
    'ME',  # Montenegro (candidato a UE)
    'MK',  # Macedonia del Norte (candidato a UE)
    'AL',  # Albania (candidato a UE)
    'BA',  # Bosnia y Herzegovina (potencial candidato)
]

# Lista completa: EU27 + europeos adicionales
paises_europa = eu27 + extra_europeos

print(f"EU27: {len(eu27)} países")
print(f"Extra europeos: {len(extra_europeos)} países")
print(f"Total Europa: {len(paises_europa)} países")
print(f"\nLista completa: {paises_europa}")

EU27: 27 países
Extra europeos: 10 países
Total Europa: 37 países

Lista completa: ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'NO', 'CH', 'IS', 'UK', 'RS', 'TR', 'ME', 'MK', 'AL', 'BA']


In [58]:
# ============================================================
# REHACER EL MERGE CON TODOS LOS PAÍSES EUROPEOS
# ============================================================
# El merge es idéntico al anterior, solo cambia el filtro final:
# en vez de filtrar por eu27, filtro por paises_europa.
# ============================================================

df_final = df_salarios_api.merge(df_pib_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_viv_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gin_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gen_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_pobreza, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_desempleo, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_inflacion, on=['geo', 'anio'], how='outer')

# Filtrar: solo países de la lista Europa (EU27 + vecinos)
df_final = df_final[df_final['geo'].isin(paises_europa)].copy()

# Ordenar por país y año
df_final = df_final.sort_values(['geo', 'anio']).reset_index(drop=True)

# Añado una columna que indica si el país es EU27 o no
# Esto será útil en el análisis para comparar miembros vs no miembros
df_final['es_eu27'] = df_final['geo'].isin(eu27)

# Verificación completa
print("=" * 60)
print("MERGE DEFINITIVO - EUROPA COMPLETA")
print("=" * 60)
print(f"Filas: {len(df_final)}")
print(f"Columnas: {len(df_final.columns)}")
print(f"Nombres: {df_final.columns.tolist()}")
print(f"Rango temporal: {df_final['anio'].min()} - {df_final['anio'].max()}")
print(f"Países totales: {df_final['geo'].nunique()}")
print(f"  - EU27: {df_final[df_final['es_eu27']]['geo'].nunique()}")
print(f"  - Extra: {df_final[~df_final['es_eu27']]['geo'].nunique()}")

dupes = df_final.groupby(['geo', 'anio']).size()
print(f"\nMáx filas por país/año: {dupes.max()} (debe ser 1)")

print(f"\nNulos por columna:")
print(df_final.isnull().sum())

print(f"\n% de nulos por columna:")
print((df_final.isnull().sum() / len(df_final) * 100).round(1))

print(f"\nPrimeras filas:")
print(df_final.head(10))

# Verificación de países presentes
print(f"\nPaíses en el dataset:")
print(sorted(df_final['geo'].unique()))

MERGE DEFINITIVO - EUROPA COMPLETA
Filas: 962
Columnas: 12
Nombres: ['geo', 'anio', 'salario_min_eur', 'salario_min_pps', 'pib_pps', 'indice_vivienda', 'gini', 'brecha_genero', 'pobreza_laboral', 'tasa_desempleo', 'inflacion_hicp', 'es_eu27']
Rango temporal: 1999 - 2026
Países totales: 37
  - EU27: 27
  - Extra: 10

Máx filas por país/año: 1 (debe ser 1)

Nulos por columna:
geo                  0
anio                 0
salario_min_eur    396
salario_min_pps    396
pib_pps            538
indice_vivienda    684
gini               659
brecha_genero      467
pobreza_laboral    258
tasa_desempleo     379
inflacion_hicp      48
es_eu27              0
dtype: int64

% de nulos por columna:
geo                 0.0
anio                0.0
salario_min_eur    41.2
salario_min_pps    41.2
pib_pps            55.9
indice_vivienda    71.1
gini               68.5
brecha_genero      48.5
pobreza_laboral    26.8
tasa_desempleo     39.4
inflacion_hicp      5.0
es_eu27             0.0
dtype: float64

Prime

In [59]:
# ============================================================
# GUARDAR DATASET DEFINITIVO
# ============================================================

df_final.to_csv('./data/dataset_europa_final.csv', index=False)

print(f"Guardado: ./data/dataset_europa_final.csv")
print(f"  {len(df_final)} filas × {len(df_final.columns)} columnas")

print(f"\n{'=' * 50}")
print(f"VERIFICACIÓN DE REQUISITOS")
print(f"{'=' * 50}")
print(f"  Filas:    {len(df_final)} {'✓' if len(df_final) >= 1000 else '✗ (necesito 1000+)'}")
print(f"  Columnas: {len(df_final.columns)} {'✓' if len(df_final.columns) >= 10 else '✗ (necesito 10+)'}")

Guardado: ./data/dataset_europa_final.csv
  962 filas × 12 columnas

VERIFICACIÓN DE REQUISITOS
  Filas:    962 ✗ (necesito 1000+)
  Columnas: 12 ✓


### Ajuste: añadir Moldova a la lista

Con 37 países obtengo 962 filas, cerca del objetivo de 1000 pero sin alcanzarlo.
Al revisar los valores únicos de la columna `geo` en los datasets descargados,
compruebo que Moldova (MD) aparece en los datos de Eurostat pero no la había incluido
en la lista inicial.

Moldova es candidata oficial a la adhesión a la UE desde junio de 2022 y firmó un
acuerdo de asociación con la UE en 2014. Incluirla encaja con la narrativa del
análisis: comparar las condiciones laborales del bloque comunitario con las de los
países que aspiran a formar parte de él.

In [63]:
# ============================================================
# LISTA AMPLIADA DE PAÍSES EUROPEOS
# ============================================================
# Añado MD (Moldavia) que también aparece en los datos de Eurostat.
# Moldavia firmó un acuerdo de asociación con la UE en 2014 y es
# candidato oficial a la adhesión desde 2022. Encaja con la narrativa
# de comparar el bloque comunitario con sus vecinos y aspirantes.
# ============================================================

eu27 = ['AT','BE','BG','CY','CZ','DE','DK','EE','EL','ES',
        'FI','FR','HR','HU','IE','IT','LT','LU','LV','MT',
        'NL','PL','PT','RO','SE','SI','SK']

extra_europeos = [
    'NO',  # Noruega (EEE, no UE)
    'CH',  # Suiza (acuerdos bilaterales con UE, no miembro)
    'IS',  # Islandia (EEE, no UE)
    'UK',  # Reino Unido (ex-UE, salió en 2020)
    'RS',  # Serbia (candidato a UE)
    'TR',  # Turquía (candidato a UE)
    'ME',  # Montenegro (candidato a UE)
    'MK',  # Macedonia del Norte (candidato a UE)
    'AL',  # Albania (candidato a UE)
    'BA',  # Bosnia y Herzegovina (potencial candidato)
    'MD',  # Moldavia (candidato a UE desde 2022)
]

paises_europa = eu27 + extra_europeos

print(f"EU27: {len(eu27)} países")
print(f"Extra europeos: {len(extra_europeos)} países")
print(f"Total Europa: {len(paises_europa)} países")

EU27: 27 países
Extra europeos: 11 países
Total Europa: 38 países


In [64]:
# ============================================================
# MERGE DEFINITIVO - EUROPA COMPLETA (con Moldova)
# ============================================================

df_final = df_salarios_api.merge(df_pib_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_viv_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gin_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gen_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_pobreza, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_desempleo, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_inflacion, on=['geo', 'anio'], how='outer')

df_final = df_final[df_final['geo'].isin(paises_europa)].copy()
df_final = df_final.sort_values(['geo', 'anio']).reset_index(drop=True)
df_final['es_eu27'] = df_final['geo'].isin(eu27)

print("=" * 60)
print("MERGE DEFINITIVO - EUROPA COMPLETA")
print("=" * 60)
print(f"Filas: {len(df_final)}")
print(f"Columnas: {len(df_final.columns)}")
print(f"Nombres: {df_final.columns.tolist()}")
print(f"Rango temporal: {df_final['anio'].min()} - {df_final['anio'].max()}")
print(f"Países totales: {df_final['geo'].nunique()}")
print(f"  - EU27: {df_final[df_final['es_eu27']]['geo'].nunique()}")
print(f"  - Extra: {df_final[~df_final['es_eu27']]['geo'].nunique()}")

dupes = df_final.groupby(['geo', 'anio']).size()
print(f"\nMáx filas por país/año: {dupes.max()} (debe ser 1)")

print(f"\nNulos por columna:")
print(df_final.isnull().sum())

print(f"\n% de nulos por columna:")
print((df_final.isnull().sum() / len(df_final) * 100).round(1))

MERGE DEFINITIVO - EUROPA COMPLETA
Filas: 962
Columnas: 12
Nombres: ['geo', 'anio', 'salario_min_eur', 'salario_min_pps', 'pib_pps', 'indice_vivienda', 'gini', 'brecha_genero', 'pobreza_laboral', 'tasa_desempleo', 'inflacion_hicp', 'es_eu27']
Rango temporal: 1999 - 2026
Países totales: 37
  - EU27: 27
  - Extra: 10

Máx filas por país/año: 1 (debe ser 1)

Nulos por columna:
geo                  0
anio                 0
salario_min_eur    396
salario_min_pps    396
pib_pps            538
indice_vivienda    684
gini               659
brecha_genero      467
pobreza_laboral    258
tasa_desempleo     379
inflacion_hicp      48
es_eu27              0
dtype: int64

% de nulos por columna:
geo                 0.0
anio                0.0
salario_min_eur    41.2
salario_min_pps    41.2
pib_pps            55.9
indice_vivienda    71.1
gini               68.5
brecha_genero      48.5
pobreza_laboral    26.8
tasa_desempleo     39.4
inflacion_hicp      5.0
es_eu27             0.0
dtype: float64


In [65]:
# ============================================================
# GUARDAR DATASET DEFINITIVO
# ============================================================

df_final.to_csv('./data/dataset_europa_final.csv', index=False)

print(f"Guardado: ./data/dataset_europa_final.csv")
print(f"  {len(df_final)} filas × {len(df_final.columns)} columnas")

print(f"\n{'=' * 50}")
print(f"VERIFICACIÓN DE REQUISITOS")
print(f"{'=' * 50}")
print(f"  Filas:    {len(df_final)} {'✓' if len(df_final) >= 1000 else '✗ (necesito 1000+)'}")
print(f"  Columnas: {len(df_final.columns)} {'✓' if len(df_final.columns) >= 10 else '✗ (necesito 10+)'}")

Guardado: ./data/dataset_europa_final.csv
  962 filas × 12 columnas

VERIFICACIÓN DE REQUISITOS
  Filas:    962 ✗ (necesito 1000+)
  Columnas: 12 ✓


### Corrección: aplicar la lista ampliada a todos los datasets

El merge sigue dando 37 países porque cuando limpié los salarios de la API,
filtré por la lista `eu27` (27 países). Los países extra como Moldova, Noruega
o Suiza quedaron descartados antes del merge.

Para solucionarlo, vuelvo al DataFrame de salarios antes de filtrar (`df_pivot`)
y rehago el cálculo de la media anual usando la lista ampliada `paises_europa`.

In [66]:
# ============================================================
# REHACER EL FILTRADO DE SALARIOS CON LA LISTA AMPLIADA
# ============================================================
# Cuando calculé la media anual de salarios, filtré por eu27.
# Ahora necesito filtrar por paises_europa para incluir los países
# adicionales (Noruega, Suiza, Moldova, etc.)
# Vuelvo al df_pivot (que tiene todos los países de la API) y rehago
# el cálculo con la lista ampliada.
# ============================================================

df_salarios_api = df_pivot.groupby(['geo', 'anio']).mean(numeric_only=True).reset_index()
df_salarios_api = df_salarios_api[df_salarios_api['geo'].isin(paises_europa)].copy()

print(f"Salarios API (Europa ampliada): {len(df_salarios_api)} filas")
print(f"Países: {sorted(df_salarios_api['geo'].unique())}")
print(f"¿Está Moldova (MD)?: {'MD' in df_salarios_api['geo'].values}")

Salarios API (Europa ampliada): 728 filas
Países: ['AL', 'BE', 'BG', 'CY', 'CZ', 'DE', 'EE', 'EL', 'ES', 'FR', 'HR', 'HU', 'IE', 'LT', 'LU', 'LV', 'MD', 'ME', 'MK', 'MT', 'NL', 'PL', 'PT', 'RO', 'RS', 'SI', 'SK', 'TR', 'UK']
¿Está Moldova (MD)?: True


In [67]:
# ============================================================
# MERGE DEFINITIVO - EUROPA COMPLETA (con lista ampliada corregida)
# ============================================================

df_final = df_salarios_api.merge(df_pib_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_viv_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gin_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_gen_long, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_pobreza, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_desempleo, on=['geo', 'anio'], how='outer')
df_final = df_final.merge(df_inflacion, on=['geo', 'anio'], how='outer')

df_final = df_final[df_final['geo'].isin(paises_europa)].copy()
df_final = df_final.sort_values(['geo', 'anio']).reset_index(drop=True)
df_final['es_eu27'] = df_final['geo'].isin(eu27)

print("=" * 60)
print("MERGE DEFINITIVO - EUROPA COMPLETA")
print("=" * 60)
print(f"Filas: {len(df_final)}")
print(f"Columnas: {len(df_final.columns)}")
print(f"Rango temporal: {df_final['anio'].min()} - {df_final['anio'].max()}")
print(f"Países totales: {df_final['geo'].nunique()}")
print(f"  - EU27: {df_final[df_final['es_eu27']]['geo'].nunique()}")
print(f"  - Extra: {df_final[~df_final['es_eu27']]['geo'].nunique()}")

dupes = df_final.groupby(['geo', 'anio']).size()
print(f"\nMáx filas por país/año: {dupes.max()} (debe ser 1)")

print(f"\nNulos por columna:")
print(df_final.isnull().sum())

MERGE DEFINITIVO - EUROPA COMPLETA
Filas: 1016
Columnas: 12
Rango temporal: 1999 - 2026
Países totales: 38
  - EU27: 27
  - Extra: 11

Máx filas por país/año: 1 (debe ser 1)

Nulos por columna:
geo                  0
anio                 0
salario_min_eur    288
salario_min_pps    319
pib_pps            592
indice_vivienda    738
gini               713
brecha_genero      521
pobreza_laboral    312
tasa_desempleo     433
inflacion_hicp     102
es_eu27              0
dtype: int64


In [68]:
# ============================================================
# GUARDAR DATASET DEFINITIVO
# ============================================================

df_final.to_csv('./data/dataset_europa_final.csv', index=False)

print(f"Guardado: ./data/dataset_europa_final.csv")
print(f"  {len(df_final)} filas × {len(df_final.columns)} columnas")

print(f"\n{'=' * 50}")
print(f"VERIFICACIÓN DE REQUISITOS")
print(f"{'=' * 50}")
print(f"  Filas:    {len(df_final)} {'✓' if len(df_final) >= 1000 else '✗ (necesito 1000+)'}")
print(f"  Columnas: {len(df_final.columns)} {'✓' if len(df_final.columns) >= 10 else '✗ (necesito 10+)'}")

Guardado: ./data/dataset_europa_final.csv
  1016 filas × 12 columnas

VERIFICACIÓN DE REQUISITOS
  Filas:    1016 ✓
  Columnas: 12 ✓


## Resumen del Notebook 1: Obtención y limpieza de datos

### Fuentes de datos
- **5 datasets TSV** descargados manualmente de la web de Eurostat (salario mínimo, PIB per cápita, precios de vivienda, coeficiente Gini, brecha de género)
- **3 datasets adicionales** descargados de la API pública de Eurostat (pobreza laboral, tasa de desempleo, inflación HICP)

### Proceso de limpieza
1. Separación de la columna compuesta de Eurostat (freq, categoría, unidad, geo)
2. Filtrado de categorías para evitar productos cartesianos en el merge (vivienda: TOTAL + I10_A_AVG; Gini: TOTAL; pobreza laboral: EMP + T + Y18-64; desempleo: T + Y15-74 + PC_ACT)
3. Conversión de formato ancho a largo con `pd.melt()`
4. Limpieza de tipos con `pd.to_numeric(errors='coerce')` para los marcadores de Eurostat (": m", "42 p")
5. Cálculo de media anual para datos semestrales (S1 + S2)
6. Merge de los 8 datasets sobre `geo + anio` con outer join

### Dataset final: `dataset_europa_final.csv`
- **1016 filas** × **12 columnas**
- **38 países europeos**: 27 EU + 11 vecinos (NO, CH, IS, UK, RS, TR, ME, MK, AL, BA, MD)
- **Rango temporal**: 1999 - 2026
- **1 fila por país y año**, sin duplicados